# Narrative Network Analysis — Northern Ireland Education Textbooks

## Research Framework
Following Bearman & Stovel (2000) *"Becoming a Nazi: A Model for Narrative Networks"*, this notebook applies narrative network analysis to compare how Protestant/Unionist (Option 1) and Catholic/Nationalist (Option 2) education textbooks structure their historical narratives.

## Analyses
1. **Descriptive Characteristics** — Node/tie type distributions, density (Bearman Table 1 & 2)
2. **Component Analysis** — Narrative fragmentation vs. coherence (Bearman Fig. 3)
3. **Centrality Analysis** — Structural importance by entity type (Bearman Table 3)
4. **Reachability** — Narrative integration curves (Bearman Fig. 5)
5. **HITS Hub & Authority** — Agent vs. focal-point framing
6. **Community Detection** — Narrative theme clusters
7. **Sentiment-Weighted Analysis** — Positive/negative subnetwork structure
8. **Cross-Corpus Synthesis** — Comparative summary

In [1]:
# Cell 1: Setup & Data Loading
import os
import sys
import glob
import pandas as pd
import numpy as np
import igraph as ig
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# ── Load triples from CSV ──────────────────────────────────────────
sys.path.insert(0, '../scripts')
csv_files = sorted(glob.glob('../graph_data/preselected_kg_*.csv'))
if not csv_files:
    raise FileNotFoundError("No CSV files found in ../graph_data/. Run extraction notebook first.")

csv_path = csv_files[-1]
print(f"Loading data from: {csv_path}")
df = pd.read_csv(csv_path)
print(f"   Total triples: {len(df)}")
print(f"   Columns: {list(df.columns)}")
print(f"   Corpora: {df['corpus'].value_counts().to_dict()}")


df_opt1 = df[df['corpus'] == 'option1'].copy()
df_opt2 = df[df['corpus'] == 'option2'].copy()

# ── Sentiment lexicon (380 predicates: all ontology + extraction data) ──
# Positive (53): constructive, cooperative, resolution-oriented
# Negative (88): adversarial, destructive, coercive
# Neutral (239): descriptive, procedural, ambiguous → default for unknown
SENTIMENT_POSITIVE = {
    'Accept',
    'Achieve',
    'Agree',
    'Allow',
    'Allowed',
    'Appreciate',
    'Assist',
    'Be Friendly Towards',
    'Be Happy',
    'Benefit',
    'Build',
    'Commemorate',
    'Compromise',
    'Cooperate',
    'Decommission',
    'Defend',
    'Encourage',
    'Enjoy',
    'Ensure',
    'Favour',
    'Favourable',
    'Favoured',
    'Glad',
    'Grant',
    'Guarantee',
    'Help',
    'Hope',
    'Improve',
    'Inspire',
    'Join',
    'Keen',
    'Modernise',
    'Offer',
    'Open',
    'Praise',
    'Protect',
    'Publish',
    'Reassure',
    'Reassured',
    'Recognise',
    'Regain',
    'Release',
    'Renounce',
    'Resolve',
    'Save',
    'Share',
    'Solve',
    'Strengthen',
    'Succeed',
    'Support',
    'Sympathetic',
    'Welcome',
    'Win',
}

SENTIMENT_NEGATIVE = {
    'Abandon',
    'Abolish',
    'Accuse',
    'Alarm',
    'Annoy',
    'Appease',
    'Argue',
    'Arrest',
    'Attack',
    'Ban',
    'Bomb',
    'Break',
    'Campaign',
    'Challenge',
    'Condemn',
    'Criticise',
    'Damage',
    'Demand',
    'Denounce',
    'Destroy',
    'Deteriorate',
    'Detonate',
    'Die',
    'Disagree',
    'Dislike',
    'Dismantle',
    'Dissatisfied',
    'Divide',
    'Dominate',
    'Emigrate',
    'Enforce',
    'Evacuate',
    'Explode',
    'Fail',
    'Fear',
    'Fight',
    'Fire',
    'Flee',
    'Force',
    'Furious',
    'Hit back',
    'Horrify',
    'Hunger strike',
    'Impose',
    'Injure',
    'Insist',
    'Intern',
    'Invade',
    'Isolate',
    'Justify',
    'Kill',
    'Lay claim',
    'Limit',
    'Lose',
    'Object',
    'Oppose',
    'Outrage',
    'Perish',
    'Persuade',
    'Protest',
    'Raid',
    'Rampage',
    'Refuse',
    'Regret',
    'Reject',
    'Remove',
    'Resent',
    'Resign',
    'Resist',
    'Retaliate',
    'Riot',
    'Search',
    'Sell',
    'Smeared',
    'Smuggle',
    'Split',
    'Steal',
    'Stop',
    'Suffer',
    'Surrender',
    'Suspect',
    'Suspend',
    'Target',
    'Tax',
    'Threaten',
    'Undermine',
    'Weaken',
    'Worsen',
}

def get_sentiment(pred):
    if pred in SENTIMENT_POSITIVE: return 'positive'
    if pred in SENTIMENT_NEGATIVE: return 'negative'
    return 'neutral'

df['sentiment'] = df['predicate'].apply(get_sentiment)
df_opt1['sentiment'] = df_opt1['predicate'].apply(get_sentiment)
df_opt2['sentiment'] = df_opt2['predicate'].apply(get_sentiment)

# ── Build igraph directed graphs ───────────────────────────────────
def build_graph(triples_df, name="graph"):
    nodes = {}
    for _, row in triples_df.iterrows():
        if row['subject'] not in nodes:
            nodes[row['subject']] = row['subject_type']
        if row['object'] not in nodes:
            nodes[row['object']] = row['object_type']

    node_list = list(nodes.keys())
    node_idx = {n: i for i, n in enumerate(node_list)}

    g = ig.Graph(directed=True)
    g.add_vertices(len(node_list))
    g.vs['name'] = node_list
    g.vs['type'] = [nodes[n] for n in node_list]

    edge_counter = defaultdict(lambda: {'count': 0, 'predicates': [], 'confidences': [], 'sentiments': []})
    for _, row in triples_df.iterrows():
        key = (row['subject'], row['object'])
        edge_counter[key]['count'] += 1
        edge_counter[key]['predicates'].append(row['predicate'])
        edge_counter[key]['confidences'].append(row['confidence'])
        edge_counter[key]['sentiments'].append(get_sentiment(row['predicate']))

    edges = []
    edge_attrs = {'predicate': [], 'sentiment': [], 'confidence': [], 'weight': []}
    for (src, tgt), info in edge_counter.items():
        edges.append((node_idx[src], node_idx[tgt]))
        edge_attrs['predicate'].append(Counter(info['predicates']).most_common(1)[0][0])
        edge_attrs['sentiment'].append(Counter(info['sentiments']).most_common(1)[0][0])
        edge_attrs['confidence'].append(np.mean(info['confidences']))
        edge_attrs['weight'].append(info['count'])

    g.add_edges(edges)
    for attr, values in edge_attrs.items():
        g.es[attr] = values
    g['name'] = name
    return g

G_opt1 = build_graph(df_opt1, "Option 1 (Protestant/Unionist)")
G_opt2 = build_graph(df_opt2, "Option 2 (Catholic/Nationalist)")
G_combined = build_graph(df, "Combined")

G_opt1_u = G_opt1.as_undirected(mode="collapse")
G_opt2_u = G_opt2.as_undirected(mode="collapse")
G_combined_u = G_combined.as_undirected(mode="collapse")

# ── Standardize entity types ───────────────────────────────────────
TYPE_MAP = {
    'Event': 'Event', 'Figure': 'Figure', 'Person': 'Figure',
    'Entity': 'Entity', 'Organisation': 'Entity', 'Place': 'Entity',
    'Object': 'Entity', 'Document': 'Entity', 'Action': 'Entity',
    'Emotions': 'Entity', 'Miscellaneous': 'Entity',
}
for g in [G_opt1, G_opt2, G_combined, G_opt1_u, G_opt2_u, G_combined_u]:
    g.vs['type_broad'] = [TYPE_MAP.get(t, 'Entity') for t in g.vs['type']]

print(f"\nGraphs built:")
print(f"   Option 1: {G_opt1.vcount()} nodes, {G_opt1.ecount()} edges")
print(f"   Option 2: {G_opt2.vcount()} nodes, {G_opt2.ecount()} edges")
print(f"   Combined: {G_combined.vcount()} nodes, {G_combined.ecount()} edges")

# ── Plotly helpers ─────────────────────────────────────────────────
CORPUS_COLORS = {'option1': '#E8913A', 'option2': '#4A90D9'}
SENTIMENT_COLORS = {'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#95a5a6'}
TYPE_COLORS = {'Event': '#e74c3c', 'Figure': '#3498db', 'Entity': '#f39c12'}

def plot_network_plotly(g, layout_algo='fr', title='Network', color_attr='type_broad',
                        size_attr=None, width=900, height=700):
    layout = g.layout(layout_algo)
    coords = np.array(layout.coords)

    if size_attr and size_attr in g.vs.attributes():
        sizes = np.array(g.vs[size_attr], dtype=float)
        sizes = 8 + 25 * (sizes - sizes.min()) / (sizes.max() - sizes.min() + 1e-9)
    else:
        sizes = np.full(g.vcount(), 10)

    color_values = g.vs[color_attr] if color_attr in g.vs.attributes() else ['Entity'] * g.vcount()
    unique_colors = sorted(set(color_values))
    if color_attr == 'type_broad':
        cmap = TYPE_COLORS
    elif color_attr == 'community':
        palette = px.colors.qualitative.Set2
        cmap = {c: palette[i % len(palette)] for i, c in enumerate(unique_colors)}
    else:
        palette = px.colors.qualitative.Plotly
        cmap = {c: palette[i % len(palette)] for i, c in enumerate(unique_colors)}
    node_colors = [cmap.get(v, '#999999') for v in color_values]

    edge_x, edge_y = [], []
    for e in g.es:
        x0, y0 = coords[e.source]
        x1, y1 = coords[e.target]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.5, color='#cccccc'), hoverinfo='none')

    node_traces = []
    for val in unique_colors:
        idxs = [i for i, v in enumerate(color_values) if v == val]
        node_traces.append(go.Scatter(
            x=coords[idxs, 0].tolist(), y=coords[idxs, 1].tolist(),
            mode='markers', name=str(val),
            marker=dict(size=[sizes[i] for i in idxs], color=cmap.get(val, '#999'),
                        line=dict(width=0.5, color='white')),
            text=[g.vs[i]['name'] for i in idxs],
            hovertemplate='<b>%{text}</b><extra>' + str(val) + '</extra>',
        ))

    fig = go.Figure(data=[edge_trace] + node_traces,
        layout=go.Layout(
            title=title, showlegend=True, hovermode='closest',
            width=width, height=height,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor='white', legend=dict(x=1.02, y=1),
        ))
    return fig

print("Setup complete!")


Loading data from: ../graph_data/preselected_kg_20260303_081411.csv
   Total triples: 1003
   Columns: ['corpus', 'subject', 'subject_type', 'predicate', 'object', 'object_type', 'confidence', 'chunk_id']
   Corpora: {'option1': 709, 'option2': 294}

Graphs built:
   Option 1: 531 nodes, 613 edges
   Option 2: 277 nodes, 277 edges
   Combined: 774 nodes, 888 edges
Setup complete!


In [2]:
# Cell 2: Descriptive Characteristics (Bearman Table 1 & 2)

print("="*70)
print("  ANALYSIS 1: DESCRIPTIVE NETWORK CHARACTERISTICS")
print("="*70)

# ── Table 1: Node type distribution ────────────────────────────────
print("\nTable 1: Node Type Distribution (cf. Bearman Table 1)")
print("-"*60)

rows = []
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2), ('Combined', G_combined)]:
    type_counts = Counter(g.vs['type_broad'])
    total = g.vcount()
    rows.append({
        'Corpus': label,
        'Event (n)': type_counts.get('Event', 0),
        'Event (%)': round(100 * type_counts.get('Event', 0) / total, 1),
        'Figure (n)': type_counts.get('Figure', 0),
        'Figure (%)': round(100 * type_counts.get('Figure', 0) / total, 1),
        'Entity (n)': type_counts.get('Entity', 0),
        'Entity (%)': round(100 * type_counts.get('Entity', 0) / total, 1),
        'N of Nodes': total,
    })

table1 = pd.DataFrame(rows)
print(table1.to_string(index=False))

# ── Node type bar chart ────────────────────────────────────────────
fig_types = make_subplots(rows=1, cols=2, subplot_titles=('Node Count', 'Node %'))
for i, corpus in enumerate(['Option 1', 'Option 2']):
    row_data = table1[table1['Corpus'] == corpus].iloc[0]
    for typ, color in TYPE_COLORS.items():
        fig_types.add_trace(go.Bar(
            x=[corpus], y=[row_data[f'{typ} (n)']],
            name=typ, marker_color=color,
            legendgroup=typ, showlegend=(i==0),
        ), row=1, col=1)
        fig_types.add_trace(go.Bar(
            x=[corpus], y=[row_data[f'{typ} (%)']],
            name=typ, marker_color=color,
            legendgroup=typ, showlegend=False,
        ), row=1, col=2)
fig_types.update_layout(title='Node Type Distribution by Corpus', barmode='group', width=800, height=400)
fig_types.show()

# ── Table 2: Tie type cross-tabulation ─────────────────────────────
print("\n\nTable 2: Tie Type Cross-Tabulation (cf. Bearman Table 2)")
print("-"*60)

for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    print(f"\n  {label}:")
    tie_types = defaultdict(int)
    for e in g.es:
        src_type = g.vs[e.source]['type_broad']
        tgt_type = g.vs[e.target]['type_broad']
        tie_types[f"{src_type} -> {tgt_type}"] += 1

    total_ties = sum(tie_types.values())
    print(f"  {'Tie Type':<25} {'n':>6} {'%':>8}")
    print(f"  {'-'*25} {'-'*6} {'-'*8}")
    for key, count in sorted(tie_types.items(), key=lambda x: -x[1]):
        print(f"  {key:<25} {count:>6} {100*count/total_ties:>7.1f}%")
    print(f"  {'N of ties':<25} {total_ties:>6}")
    print(f"  {'Density':<25} {g.density():>6.4f}")

# ── Tie type heatmap ───────────────────────────────────────────────
types_order = ['Figure', 'Event', 'Entity']
fig_heat = make_subplots(rows=1, cols=2, subplot_titles=('Option 1', 'Option 2'),
                         horizontal_spacing=0.15)
for col_idx, (label, g) in enumerate([('Option 1', G_opt1), ('Option 2', G_opt2)], 1):
    matrix = pd.DataFrame(0, index=types_order, columns=types_order)
    for e in g.es:
        src_t = g.vs[e.source]['type_broad']
        tgt_t = g.vs[e.target]['type_broad']
        if src_t in types_order and tgt_t in types_order:
            matrix.loc[src_t, tgt_t] += 1
    fig_heat.add_trace(go.Heatmap(
        z=matrix.values, x=types_order, y=types_order,
        text=matrix.values, texttemplate='%{text}',
        colorscale='YlOrRd', showscale=(col_idx==2),
    ), row=1, col=col_idx)
fig_heat.update_layout(title='Tie Type Cross-Tabulation (Source -> Target)', width=800, height=400)
fig_heat.show()

# ── Basic metrics ──────────────────────────────────────────────────
print("\n\nBasic Network Metrics Comparison")
print("-"*60)
metrics = []
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2), ('Combined', G_combined)]:
    degrees = g.degree()
    metrics.append({
        'Corpus': label, 'Nodes': g.vcount(), 'Edges': g.ecount(),
        'Density': round(g.density(), 4), 'Avg Degree': round(np.mean(degrees), 2),
        'Max Degree': max(degrees), 'Reciprocity': round(g.reciprocity(), 3),
    })
print(pd.DataFrame(metrics).to_string(index=False))


  ANALYSIS 1: DESCRIPTIVE NETWORK CHARACTERISTICS

Table 1: Node Type Distribution (cf. Bearman Table 1)
------------------------------------------------------------
  Corpus  Event (n)  Event (%)  Figure (n)  Figure (%)  Entity (n)  Entity (%)  N of Nodes
Option 1         76       14.3          29         5.5         426        80.2         531
Option 2         56       20.2          20         7.2         201        72.6         277
Combined        127       16.4          49         6.3         598        77.3         774




Table 2: Tie Type Cross-Tabulation (cf. Bearman Table 2)
------------------------------------------------------------

  Option 1:
  Tie Type                       n        %
  ------------------------- ------ --------
  Entity -> Entity             329    53.7%
  Figure -> Entity              96    15.7%
  Event -> Entity               71    11.6%
  Entity -> Event               69    11.3%
  Figure -> Event               29     4.7%
  Event -> Event                 8     1.3%
  Figure -> Figure               7     1.1%
  Entity -> Figure               4     0.7%
  N of ties                    613
  Density                   0.0022

  Option 2:
  Tie Type                       n        %
  ------------------------- ------ --------
  Entity -> Entity             114    41.2%
  Entity -> Event               55    19.9%
  Event -> Entity               43    15.5%
  Figure -> Entity              27     9.7%
  Figure -> Event               19     6.9%
  Event -> Event                12  



Basic Network Metrics Comparison
------------------------------------------------------------
  Corpus  Nodes  Edges  Density  Avg Degree  Max Degree  Reciprocity
Option 1    531    613   0.0022        2.31          56        0.056
Option 2    277    277   0.0036        2.00          23        0.030
Combined    774    888   0.0015        2.29          56        0.050


In [3]:
# Cell 3: Component Analysis (Bearman Fig. 3)

print("="*70)
print("  ANALYSIS 2: COMPONENT ANALYSIS")
print("="*70)

comp_data = []
for label, g in [('Option 1', G_opt1_u), ('Option 2', G_opt2_u)]:
    components = g.connected_components()
    sizes = sorted([len(c) for c in components], reverse=True)
    giant = sizes[0]
    total = g.vcount()

    print(f"\n  {label}:")
    print(f"    Total components: {len(sizes)}")
    print(f"    Giant component: {giant} nodes ({100*giant/total:.1f}% of network)")
    print(f"    Isolates (size=1): {sizes.count(1)}")

    size_counts = Counter(sizes)
    print(f"\n    {'Size':<10} {'Count':<10} {'% of nodes':<15}")
    print(f"    {'-'*10} {'-'*10} {'-'*15}")
    for size in sorted(size_counts.keys()):
        count = size_counts[size]
        pct = 100 * size * count / total
        if size <= 10 or size == giant:
            print(f"    {size:<10} {count:<10} {pct:.1f}%")

    for size, count in size_counts.items():
        comp_data.append({
            'corpus': label, 'component_size': size,
            'count': count, 'pct_nodes': 100 * size * count / total
        })

# ── Bar chart ──────────────────────────────────────────────────────
comp_df = pd.DataFrame(comp_data)
fig_comp = make_subplots(rows=1, cols=2,
    subplot_titles=('Component Count by Size', '% of Nodes by Component Size'))
for corpus, color in [('Option 1', CORPUS_COLORS['option1']), ('Option 2', CORPUS_COLORS['option2'])]:
    subset = comp_df[comp_df['corpus'] == corpus]
    subset_small = subset[subset['component_size'] <= 10].copy()
    large = subset[subset['component_size'] > 10]
    if not large.empty:
        large_row = pd.DataFrame([{
            'corpus': corpus, 'component_size': '>10',
            'count': large['count'].sum(), 'pct_nodes': large['pct_nodes'].sum()
        }])
        subset_small = pd.concat([subset_small, large_row])

    fig_comp.add_trace(go.Bar(
        x=[str(s) for s in subset_small['component_size']],
        y=subset_small['count'], name=corpus, marker_color=color, legendgroup=corpus,
    ), row=1, col=1)
    fig_comp.add_trace(go.Bar(
        x=[str(s) for s in subset_small['component_size']],
        y=subset_small['pct_nodes'], name=corpus, marker_color=color,
        legendgroup=corpus, showlegend=False,
    ), row=1, col=2)

fig_comp.update_layout(title='Component Size Distribution (cf. Bearman Fig. 3)',
                       barmode='group', width=900, height=400)
fig_comp.update_xaxes(title_text='Component Size')
fig_comp.update_yaxes(title_text='Count', row=1, col=1)
fig_comp.update_yaxes(title_text='% of Nodes', row=1, col=2)
fig_comp.show()


  ANALYSIS 2: COMPONENT ANALYSIS

  Option 1:
    Total components: 58
    Giant component: 389 nodes (73.3% of network)
    Isolates (size=1): 0

    Size       Count      % of nodes     
    ---------- ---------- ---------------
    2          37         13.9%
    3          14         7.9%
    4          4          3.0%
    5          2          1.9%
    389        1          73.3%

  Option 2:
    Total components: 41
    Giant component: 177 nodes (63.9% of network)
    Isolates (size=1): 0

    Size       Count      % of nodes     
    ---------- ---------- ---------------
    2          29         20.9%
    3          5          5.4%
    4          4          5.8%
    5          1          1.8%
    6          1          2.2%
    177        1          63.9%


In [4]:
# Cell 4: Centrality Analysis (Bearman Table 3)

print("="*70)
print("  ANALYSIS 3: CENTRALITY ANALYSIS")
print("="*70)

centrality_results = {}

for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    n = g.vcount()
    in_deg = np.array(g.indegree()) / (n - 1) if n > 1 else np.zeros(n)
    out_deg = np.array(g.outdegree()) / (n - 1) if n > 1 else np.zeros(n)
    total_deg = np.array(g.degree()) / (2 * (n - 1)) if n > 1 else np.zeros(n)

    betw = np.array(g.betweenness(directed=True))
    betw_norm = betw / ((n-1)*(n-2)) if n > 2 else betw

    g_u = g.as_undirected(mode="collapse")
    try:
        eigen = np.array(g_u.eigenvector_centrality())
    except:
        eigen = np.zeros(n)

    centrality_results[label] = {
        'names': g.vs['name'], 'types': g.vs['type_broad'],
        'in_degree': in_deg, 'out_degree': out_deg,
        'total_degree': total_deg, 'betweenness': betw_norm, 'eigenvector': eigen,
    }

# ── Mean centrality by type ────────────────────────────────────────
print("\nTable 3: Mean Centrality by Entity Type (cf. Bearman Table 3)")
print("-"*70)
for label in ['Option 1', 'Option 2']:
    res = centrality_results[label]
    df_cent = pd.DataFrame({
        'type': res['types'], 'degree': res['total_degree'],
        'betweenness': res['betweenness'], 'eigenvector': res['eigenvector'],
    })
    print(f"\n  {label}:")
    grouped = df_cent.groupby('type').agg(
        n=('type', 'count'), mean_degree=('degree', 'mean'),
        mean_betweenness=('betweenness', 'mean'), mean_eigenvector=('eigenvector', 'mean'),
    ).round(4)
    print(grouped.to_string())

# ── Relative centrality matrix ─────────────────────────────────────
print("\n\nRelative Centrality Matrix (Bearman Table 3 off-diagonal)")
print("  Values = ratio of row group centrality to column group centrality")
print("-"*70)
for label in ['Option 1', 'Option 2']:
    res = centrality_results[label]
    df_cent = pd.DataFrame({'type': res['types'], 'eigenvector': res['eigenvector']})
    means = df_cent.groupby('type')['eigenvector'].mean()
    types_present = sorted(means.index.tolist())
    matrix = pd.DataFrame(index=types_present, columns=types_present, dtype=float)
    for t1 in types_present:
        for t2 in types_present:
            if t1 == t2:
                matrix.loc[t1, t2] = round(means[t1], 4)
            else:
                matrix.loc[t1, t2] = round(means[t1] / means[t2], 2) if means[t2] > 0 else np.inf
    print(f"\n  {label} (diagonal = mean eigenvector centrality):")
    print(matrix.to_string())

# ── Top 10 ─────────────────────────────────────────────────────────
print("\n\nTop 10 Most Central Entities")
print("-"*70)
for label in ['Option 1', 'Option 2']:
    res = centrality_results[label]
    df_top = pd.DataFrame({
        'Entity': res['names'], 'Type': res['types'],
        'Degree': res['total_degree'], 'Betweenness': res['betweenness'],
        'Eigenvector': res['eigenvector'],
    }).sort_values('Eigenvector', ascending=False).head(10)
    print(f"\n  {label} (ranked by eigenvector centrality):")
    print(df_top.to_string(index=False))

# ── Bar chart ──────────────────────────────────────────────────────
fig_cent = make_subplots(rows=1, cols=2,
    subplot_titles=('Option 1 - Top 10 by Eigenvector', 'Option 2 - Top 10 by Eigenvector'))
for col_idx, label in enumerate(['Option 1', 'Option 2'], 1):
    res = centrality_results[label]
    df_top = pd.DataFrame({
        'Entity': res['names'], 'Type': res['types'], 'Eigenvector': res['eigenvector'],
    }).sort_values('Eigenvector', ascending=True).tail(10)
    colors = [TYPE_COLORS.get(t, '#999') for t in df_top['Type']]
    fig_cent.add_trace(go.Bar(
        x=df_top['Eigenvector'], y=df_top['Entity'],
        orientation='h', marker_color=colors, showlegend=False,
    ), row=1, col=col_idx)
fig_cent.update_layout(title='Top 10 Entities by Eigenvector Centrality', width=1000, height=500)
fig_cent.show()


  ANALYSIS 3: CENTRALITY ANALYSIS

Table 3: Mean Centrality by Entity Type (cf. Bearman Table 3)
----------------------------------------------------------------------

  Option 1:
          n  mean_degree  mean_betweenness  mean_eigenvector
type                                                        
Entity  426       0.0020            0.0005            0.0412
Event    76       0.0023            0.0005            0.0578
Figure   29       0.0047            0.0021            0.0783

  Option 2:
          n  mean_degree  mean_betweenness  mean_eigenvector
type                                                        
Entity  201       0.0032            0.0001            0.0430
Event    56       0.0046            0.0001            0.0751
Figure   20       0.0053            0.0001            0.0334


Relative Centrality Matrix (Bearman Table 3 off-diagonal)
  Values = ratio of row group centrality to column group centrality
--------------------------------------------------------------------

In [5]:
# Cell 5: Reachability / Narrative Integration (Bearman Fig. 5)

print("="*70)
print("  ANALYSIS 4: REACHABILITY / NARRATIVE INTEGRATION")
print("="*70)

def compute_reachability_curve(g, max_steps=25):
    n = g.vcount()
    fractions_at_step = np.zeros(max_steps + 1)
    for v in range(n):
        distances = g.shortest_paths(source=v, mode="out")[0]
        for step in range(1, max_steps + 1):
            reachable = sum(1 for d in distances if d <= step and d < float('inf'))
            fractions_at_step[step] += reachable / n
    fractions_at_step /= n
    return fractions_at_step

print("\nComputing reachability curves (may take a moment)...")
max_steps = 20
reach_opt1 = compute_reachability_curve(G_opt1, max_steps)
reach_opt2 = compute_reachability_curve(G_opt2, max_steps)

print(f"\nReachability Summary (cf. Bearman Fig. 5):")
print(f"   Option 1 at {max_steps} steps: {reach_opt1[max_steps]:.3f}")
print(f"   Option 2 at {max_steps} steps: {reach_opt2[max_steps]:.3f}")

steps = list(range(1, max_steps + 1))
fig_reach = go.Figure()
fig_reach.add_trace(go.Scatter(
    x=steps, y=reach_opt1[1:max_steps+1],
    mode='lines+markers', name='Option 1 (Protestant/Unionist)',
    line=dict(color=CORPUS_COLORS['option1'], width=2),
))
fig_reach.add_trace(go.Scatter(
    x=steps, y=reach_opt2[1:max_steps+1],
    mode='lines+markers', name='Option 2 (Catholic/Nationalist)',
    line=dict(color=CORPUS_COLORS['option2'], width=2),
))
fig_reach.update_layout(
    title='Mean Fraction of Elements Reached at n Steps (cf. Bearman Fig. 5)',
    xaxis_title='n of steps', yaxis_title='Fraction of elements',
    width=800, height=500, yaxis=dict(range=[0, 1]),
    legend=dict(x=0.5, y=0.15),
)
fig_reach.show()


  ANALYSIS 4: REACHABILITY / NARRATIVE INTEGRATION

Computing reachability curves (may take a moment)...

Reachability Summary (cf. Bearman Fig. 5):
   Option 1 at 20 steps: 0.084
   Option 2 at 20 steps: 0.017


In [6]:
# Cell 6: HITS - Hub & Authority Analysis

print("="*70)
print("  ANALYSIS 5: HUB & AUTHORITY (HITS)")
print("="*70)

hits_results = {}
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    hub_scores = g.hub_score()
    auth_scores = g.authority_score()
    hits_results[label] = {
        'names': g.vs['name'], 'types': g.vs['type_broad'],
        'hub': np.array(hub_scores), 'authority': np.array(auth_scores),
    }

# ── Top Hubs and Authorities ──────────────────────────────────────
for label in ['Option 1', 'Option 2']:
    res = hits_results[label]
    df_hits = pd.DataFrame({
        'Entity': res['names'], 'Type': res['types'],
        'Hub': res['hub'], 'Authority': res['authority'],
    })
    print(f"\n  {label} - Top 10 HUBS (narrative drivers / agents):")
    print(df_hits.sort_values('Hub', ascending=False).head(10)[['Entity','Type','Hub']].to_string(index=False))
    print(f"\n  {label} - Top 10 AUTHORITIES (narrative focal points):")
    print(df_hits.sort_values('Authority', ascending=False).head(10)[['Entity','Type','Authority']].to_string(index=False))

# ── Cross-corpus role comparison ───────────────────────────────────
print("\n\nCross-Corpus Hub/Authority Comparison (Shared Entities)")
print("-"*70)

opt1_names = set(hits_results['Option 1']['names'])
opt2_names = set(hits_results['Option 2']['names'])
shared = opt1_names & opt2_names

if shared:
    rows = []
    for entity in shared:
        idx1 = list(hits_results['Option 1']['names']).index(entity)
        idx2 = list(hits_results['Option 2']['names']).index(entity)
        h1 = hits_results['Option 1']['hub'][idx1]
        a1 = hits_results['Option 1']['authority'][idx1]
        h2 = hits_results['Option 2']['hub'][idx2]
        a2 = hits_results['Option 2']['authority'][idx2]
        role1 = 'Hub' if h1 > a1 else 'Authority'
        role2 = 'Hub' if h2 > a2 else 'Authority'
        rows.append({
            'Entity': entity,
            'Opt1_Hub': round(h1, 4), 'Opt1_Auth': round(a1, 4), 'Opt1_Role': role1,
            'Opt2_Hub': round(h2, 4), 'Opt2_Auth': round(a2, 4), 'Opt2_Role': role2,
            'Role_Change': role1 != role2,
        })

    df_shared = pd.DataFrame(rows).sort_values('Opt1_Hub', ascending=False)
    role_changes = df_shared[df_shared['Role_Change']]
    if not role_changes.empty:
        print(f"\n  Role REVERSALS (Hub<->Authority across corpora):")
        print(role_changes[['Entity','Opt1_Role','Opt1_Hub','Opt1_Auth',
                           'Opt2_Role','Opt2_Hub','Opt2_Auth']].to_string(index=False))

    print(f"\n  Top 15 shared entities by Hub score:")
    print(df_shared.head(15)[['Entity','Opt1_Hub','Opt1_Auth','Opt1_Role',
                               'Opt2_Hub','Opt2_Auth','Opt2_Role']].to_string(index=False))

# ── Scatter plot ───────────────────────────────────────────────────
fig_hits = make_subplots(rows=1, cols=2, subplot_titles=('Option 1', 'Option 2'))
for col, label in enumerate(['Option 1', 'Option 2'], 1):
    res = hits_results[label]
    colors = [TYPE_COLORS.get(t, '#999') for t in res['types']]
    fig_hits.add_trace(go.Scatter(
        x=list(res['hub']), y=list(res['authority']), mode='markers',
        marker=dict(size=8, color=colors, line=dict(width=0.5, color='white')),
        text=res['names'],
        hovertemplate='<b>%{text}</b><br>Hub: %{x:.4f}<br>Auth: %{y:.4f}<extra></extra>',
        showlegend=False,
    ), row=1, col=col)
fig_hits.update_xaxes(title_text='Hub Score')
fig_hits.update_yaxes(title_text='Authority Score', row=1, col=1)
fig_hits.update_layout(title='HITS: Hub vs Authority Scores', width=1000, height=500)
fig_hits.show()


  ANALYSIS 5: HUB & AUTHORITY (HITS)

  Option 1 - Top 10 HUBS (narrative drivers / agents):
           Entity   Type      Hub
  Éamon de Valera Figure 1.000000
             Éire Entity 0.392750
 Irish Free State Entity 0.308215
          Britain Entity 0.290058
Winston Churchill Figure 0.182471
           Ulster Entity 0.166355
         Unionist Entity 0.150712
           Hitler Figure 0.139549
          Germany Entity 0.131250
              IRA Entity 0.125879

  Option 1 - Top 10 AUTHORITIES (narrative focal points):
               Entity   Type  Authority
              Britain Entity   1.000000
                 Éire Entity   0.979571
         World War II  Event   0.944910
  Republic of Ireland Entity   0.784911
           Neutrality  Event   0.766744
   Anglo-Irish Treaty  Event   0.741883
     Irish Free State Entity   0.726956
          Treaty Port Entity   0.660412
Bunreacht na hÉireann Entity   0.584920
            Emergency  Event   0.577169

  Option 2 - Top 10 HUBS (narrati

In [7]:
# Cell 7: Community Detection / Clustering
# Methods: Leiden (primary, resolution-tuned), Infomap (directed), Louvain (baseline)

print("="*70)
print("  ANALYSIS 6: COMMUNITY DETECTION")
print("="*70)

# ── Method 1: Leiden with resolution sweep ────────────────────────
# Find resolution that yields 5-8 interpretable communities
print("\n--- Leiden Algorithm (resolution-tuned) ---")

def leiden_sweep(g_u, target_range=(5, 8), res_min=0.3, res_max=2.0, steps=20):
    """Sweep resolution parameter to find target number of communities."""
    best = None
    results = []
    for res in np.linspace(res_min, res_max, steps):
        comm = g_u.community_leiden(
            objective_function='modularity',
            resolution=res,
            n_iterations=10,
        )
        n = max(comm.membership) + 1
        # Only count clusters with size >= 2
        from collections import Counter as C
        sizes = C(comm.membership)
        n_real = sum(1 for s in sizes.values() if s >= 2)
        results.append((res, n, n_real, comm.modularity, comm))
    
    # Pick resolution closest to target range midpoint
    target_mid = (target_range[0] + target_range[1]) / 2
    in_range = [(r, n, nr, mod, c) for r, n, nr, mod, c in results
                if target_range[0] <= nr <= target_range[1]]
    
    if in_range:
        best = min(in_range, key=lambda x: abs(x[2] - target_mid))
    else:
        best = min(results, key=lambda x: abs(x[2] - target_mid))
    
    return best, results

leiden_results = {}
for label, g_u in [('Option 1', G_opt1_u), ('Option 2', G_opt2_u)]:
    best, sweep = leiden_sweep(g_u)
    res, n_total, n_real, mod, comm = best
    leiden_results[label] = {
        'resolution': res, 'communities': comm,
        'membership': comm.membership, 'modularity': mod,
        'n_communities': n_real, 'sweep': sweep,
    }
    print(f"\n  {label}:")
    print(f"    Resolution: {res:.3f}")
    print(f"    Modularity: {mod:.4f}")
    print(f"    Communities (size >= 2): {n_real}")
    
    # Show sweep summary
    print(f"    Resolution sweep:")
    for r, n, nr, m, _ in sweep[::4]:
        print(f"      res={r:.2f} -> {nr} communities (modularity={m:.4f})")

# ── Method 2: Infomap (directed graph) ───────────────────────────
print("\n\n--- Infomap Algorithm (directed) ---")

infomap_results = {}
for label, g_d in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    comm = g_d.community_infomap()
    membership = comm.membership
    n_total = max(membership) + 1
    sizes = Counter(membership)
    n_real = sum(1 for s in sizes.values() if s >= 2)
    
    infomap_results[label] = {
        'communities': comm, 'membership': membership,
        'modularity': comm.modularity, 'n_communities': n_real,
        'codelength': comm.codelength,
    }
    print(f"\n  {label}:")
    print(f"    Codelength: {comm.codelength:.4f}")
    print(f"    Modularity: {comm.modularity:.4f}")
    print(f"    Communities (size >= 2): {n_real}")

# ── Method 3: Louvain baseline ───────────────────────────────────
print("\n\n--- Louvain Algorithm (baseline) ---")

louvain_results = {}
for label, g_u in [('Option 1', G_opt1_u), ('Option 2', G_opt2_u)]:
    comm = g_u.community_multilevel()
    membership = comm.membership
    n_total = max(membership) + 1
    sizes = Counter(membership)
    n_real = sum(1 for s in sizes.values() if s >= 2)
    louvain_results[label] = {
        'communities': comm, 'membership': membership,
        'modularity': comm.modularity, 'n_communities': n_real,
    }
    print(f"\n  {label}: {n_real} communities, modularity={comm.modularity:.4f}")

# ── Comparison table ─────────────────────────────────────────────
print("\n\n--- Method Comparison ---")
print(f"  {'Method':<12} {'Corpus':<12} {'Communities':<14} {'Modularity':<12} {'Notes'}")
print(f"  {'-'*12} {'-'*12} {'-'*14} {'-'*12} {'-'*20}")
for label in ['Option 1', 'Option 2']:
    lr = leiden_results[label]
    ir = infomap_results[label]
    lo = louvain_results[label]
    print(f"  {'Leiden':<12} {label:<12} {lr['n_communities']:<14} {lr['modularity']:<12.4f} res={lr['resolution']:.3f}")
    print(f"  {'Infomap':<12} {label:<12} {ir['n_communities']:<14} {ir['modularity']:<12.4f} directed, codelength={ir['codelength']:.3f}")
    print(f"  {'Louvain':<12} {label:<12} {lo['n_communities']:<14} {lo['modularity']:<12.4f} baseline")
    print()

# ── Use Leiden as primary method ─────────────────────────────────
# (stored in community_results for downstream compatibility)
community_results = {}
for label, g_u, g_d in [('Option 1', G_opt1_u, G_opt1), ('Option 2', G_opt2_u, G_opt2)]:
    lr = leiden_results[label]
    membership = lr['membership']
    g_u.vs['community'] = membership
    g_d.vs['community'] = membership
    
    community_results[label] = {
        'membership': membership,
        'modularity': lr['modularity'],
        'n_communities': lr['n_communities'],
        'communities': lr['communities'],
    }
    
    # Also store Infomap as secondary attribute
    g_d.vs['community_infomap'] = infomap_results[label]['membership']

# ── Cluster detail table (Leiden) ─────────────────────────────────
print("\n\n--- Leiden Cluster Details ---")
for label, g_u, g_d in [('Option 1', G_opt1_u, G_opt1), ('Option 2', G_opt2_u, G_opt2)]:
    membership = community_results[label]['membership']
    n_communities = community_results[label]['n_communities']
    
    print(f"\n  {label} (resolution={leiden_results[label]['resolution']:.3f}):")
    print(f"    {'Cluster':<10} {'Size':<8} {'Entity Types':<30} {'Key Members'}")
    print(f"    {'-'*10} {'-'*8} {'-'*30} {'-'*30}")
    
    for c in range(max(membership) + 1):
        members = [i for i, m in enumerate(membership) if m == c]
        size = len(members)
        if size < 2:
            continue
        type_counts = Counter(g_u.vs[i]['type_broad'] for i in members)
        type_str = ', '.join(f"{t}:{n}" for t, n in type_counts.most_common(3))
        degrees_in_cluster = [(g_u.vs[i]['name'], g_u.degree(i)) for i in members]
        top_members = sorted(degrees_in_cluster, key=lambda x: -x[1])[:3]
        member_str = ', '.join(f"{name}" for name, _ in top_members)
        print(f"    {c:<10} {size:<8} {type_str:<30} {member_str}")

# ── Cross-corpus cluster alignment ─────────────────────────────────
print("\n\nCross-Corpus Cluster Alignment (Leiden)")
print("  Do the same entities cluster together in both corpora?")
print("-"*70)

opt1_names = set(G_opt1.vs['name'])
opt2_names = set(G_opt2.vs['name'])
shared = opt1_names & opt2_names

if shared:
    cluster_pairs = defaultdict(list)
    for entity in shared:
        idx1 = G_opt1.vs['name'].index(entity)
        idx2 = G_opt2.vs['name'].index(entity)
        c1 = G_opt1.vs[idx1]['community']
        c2 = G_opt2.vs[idx2]['community']
        cluster_pairs[(c1, c2)].append(entity)

    print(f"\n  Shared entities: {len(shared)}")
    print(f"  Cluster pair combinations: {len(cluster_pairs)}")
    print(f"\n  {'Opt1 Clust':<12} {'Opt2 Clust':<12} {'Count':<8} {'Entities'}")
    print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*40}")
    for (c1, c2), entities in sorted(cluster_pairs.items(), key=lambda x: -len(x[1])):
        ent_str = ', '.join(entities[:5])
        if len(entities) > 5:
            ent_str += f"... (+{len(entities)-5})"
        print(f"  {c1:<12} {c2:<12} {len(entities):<8} {ent_str}")

# ── Network visualizations (Leiden communities) ──────────────────
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    fig = plot_network_plotly(g, layout_algo='fr',
        title=f'{label} - Communities (Leiden, res={leiden_results[label]["resolution"]:.2f})',
        color_attr='community')
    fig.show()


  ANALYSIS 6: COMMUNITY DETECTION

--- Leiden Algorithm (resolution-tuned) ---

  Option 1:
    Resolution: 0.300
    Modularity: 0.8341
    Communities (size >= 2): 61
    Resolution sweep:
      res=0.30 -> 61 communities (modularity=0.8341)
      res=0.66 -> 65 communities (modularity=0.7816)
      res=1.02 -> 69 communities (modularity=0.7542)
      res=1.37 -> 73 communities (modularity=0.7321)
      res=1.73 -> 76 communities (modularity=0.7138)

  Option 2:
    Resolution: 0.300
    Modularity: 0.8771
    Communities (size >= 2): 45
    Resolution sweep:
      res=0.30 -> 45 communities (modularity=0.8771)
      res=0.66 -> 48 communities (modularity=0.8393)
      res=1.02 -> 50 communities (modularity=0.8119)
      res=1.37 -> 53 communities (modularity=0.7884)
      res=1.73 -> 57 communities (modularity=0.7669)


--- Infomap Algorithm (directed) ---

  Option 1:
    Codelength: 4.4755
    Modularity: 0.6939
    Communities (size >= 2): 107

  Option 2:
    Codelength: 3.7083


In [8]:
# Cell 8: Sentiment-Weighted Network Analysis

print("="*70)
print("  ANALYSIS 7: SENTIMENT-WEIGHTED NETWORK")
print("="*70)

def build_sentiment_subgraph(g, sentiment_type):
    edge_ids = [e.index for e in g.es if e['sentiment'] == sentiment_type]
    return g.subgraph_edges(edge_ids, delete_vertices=False)

sentiment_analysis = {}
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    results = {}
    for sent in ['positive', 'negative', 'neutral']:
        sg = build_sentiment_subgraph(g, sent)
        non_isolated = [v.index for v in sg.vs if sg.degree(v.index) > 0]
        sg_clean = sg.induced_subgraph(non_isolated) if non_isolated else sg
        results[sent] = {
            'edges': sg.ecount(), 'active_nodes': len(non_isolated),
            'density': sg_clean.density() if sg_clean.vcount() > 1 else 0,
            'subgraph': sg_clean,
        }
    sentiment_analysis[label] = results
    total_edges = g.ecount()
    print(f"\n  {label}:")
    print(f"    {'Sentiment':<12} {'Edges':<8} {'% of total':<12} {'Active nodes':<14} {'Density'}")
    print(f"    {'-'*12} {'-'*8} {'-'*12} {'-'*14} {'-'*8}")
    for sent in ['positive', 'negative', 'neutral']:
        r = results[sent]
        pct = 100 * r['edges'] / total_edges if total_edges > 0 else 0
        print(f"    {sent:<12} {r['edges']:<8} {pct:<11.1f}% {r['active_nodes']:<14} {r['density']:.4f}")

# ── Sentiment centrality ──────────────────────────────────────────
print("\n\nSentiment Centrality: Who is central in positive vs negative networks?")
print("-"*70)
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    pos_sg = sentiment_analysis[label]['positive']['subgraph']
    neg_sg = sentiment_analysis[label]['negative']['subgraph']
    pos_deg = {pos_sg.vs[i]['name']: pos_sg.degree(i) for i in range(pos_sg.vcount()) if pos_sg.degree(i) > 0}
    neg_deg = {neg_sg.vs[i]['name']: neg_sg.degree(i) for i in range(neg_sg.vcount()) if neg_sg.degree(i) > 0}
    all_entities = set(pos_deg.keys()) | set(neg_deg.keys())
    rows = []
    for entity in all_entities:
        pd_val = pos_deg.get(entity, 0)
        nd_val = neg_deg.get(entity, 0)
        total = pd_val + nd_val
        sentiment_ratio = (pd_val - nd_val) / total if total > 0 else 0
        rows.append({
            'Entity': entity, 'Positive_Degree': pd_val, 'Negative_Degree': nd_val,
            'Sentiment_Ratio': round(sentiment_ratio, 3), 'Total': total,
        })
    df_sent = pd.DataFrame(rows).sort_values('Total', ascending=False)
    print(f"\n  {label} - Top 15 entities by sentiment network participation:")
    print(df_sent.head(15).to_string(index=False))

# ── Bar chart ──────────────────────────────────────────────────────
fig_sent = go.Figure()
for label in ['Option 1', 'Option 2']:
    results = sentiment_analysis[label]
    total = sum(r['edges'] for r in results.values())
    for sent, scolor in SENTIMENT_COLORS.items():
        fig_sent.add_trace(go.Bar(
            x=[label], y=[100 * results[sent]['edges'] / total if total > 0 else 0],
            name=sent, marker_color=scolor,
            legendgroup=sent, showlegend=(label == 'Option 1'),
        ))
fig_sent.update_layout(title='Edge Sentiment Distribution by Corpus (%)',
                       barmode='stack', width=600, height=400, yaxis_title='% of edges')
fig_sent.show()


# ── Cross-Corpus Sentiment Comparison (shared entities) ──────────
print("\n\n" + "="*70)
print("  CROSS-CORPUS SENTIMENT COMPARISON")
print("="*70)

def compute_entity_sentiment(g, sentiment_analysis_result):
    """Compute per-entity sentiment stats from pre-built subgraphs."""
    pos_sg = sentiment_analysis_result['positive']['subgraph']
    neg_sg = sentiment_analysis_result['negative']['subgraph']
    neu_sg = sentiment_analysis_result['neutral']['subgraph']

    stats = {}
    for v in g.vs:
        name = v['name']
        pd_val = 0
        nd_val = 0
        nu_val = 0
        # Positive subgraph
        try:
            idx = pos_sg.vs.find(name=name).index
            pd_val = pos_sg.degree(idx)
        except ValueError:
            pass
        # Negative subgraph
        try:
            idx = neg_sg.vs.find(name=name).index
            nd_val = neg_sg.degree(idx)
        except ValueError:
            pass
        # Neutral subgraph
        try:
            idx = neu_sg.vs.find(name=name).index
            nu_val = neu_sg.degree(idx)
        except ValueError:
            pass
        total_pn = pd_val + nd_val
        ratio = (pd_val - nd_val) / total_pn if total_pn > 0 else np.nan
        stats[name] = {
            'pos': pd_val, 'neg': nd_val, 'neu': nu_val,
            'total': pd_val + nd_val + nu_val,
            'total_pn': total_pn, 'ratio': ratio,
        }
    return stats

sent_opt1 = compute_entity_sentiment(G_opt1, sentiment_analysis['Option 1'])
sent_opt2 = compute_entity_sentiment(G_opt2, sentiment_analysis['Option 2'])

# Find shared entities (present in both graphs)
shared_entities = set(sent_opt1.keys()) & set(sent_opt2.keys())
# Keep only those with at least 1 positive or negative edge in either corpus
shared_entities = {e for e in shared_entities
                   if sent_opt1[e]['total_pn'] > 0 or sent_opt2[e]['total_pn'] > 0}

print(f"\n  Shared entities with sentiment data: {len(shared_entities)}")

cross_rows = []
for entity in shared_entities:
    s1 = sent_opt1[entity]
    s2 = sent_opt2[entity]
    cross_rows.append({
        'Entity': entity,
        'Opt1 Pos': s1['pos'], 'Opt1 Neg': s1['neg'], 'Opt1 Ratio': round(s1['ratio'], 3) if not np.isnan(s1['ratio']) else None,
        'Opt2 Pos': s2['pos'], 'Opt2 Neg': s2['neg'], 'Opt2 Ratio': round(s2['ratio'], 3) if not np.isnan(s2['ratio']) else None,
    })

cross_sent_df = pd.DataFrame(cross_rows)

# Compute sentiment shift: Opt2 ratio - Opt1 ratio
cross_sent_df['Shift'] = cross_sent_df.apply(
    lambda r: round(r['Opt2 Ratio'] - r['Opt1 Ratio'], 3)
    if r['Opt1 Ratio'] is not None and r['Opt2 Ratio'] is not None
    else None, axis=1)

# Sort by absolute shift (biggest framing differences first)
cross_sent_df['Abs_Shift'] = cross_sent_df['Shift'].abs()
cross_sent_df = cross_sent_df.sort_values('Abs_Shift', ascending=False).drop(columns='Abs_Shift')

print("\n  Cross-Corpus Sentiment Comparison (sorted by largest framing difference):")
print(cross_sent_df.to_string(index=False))

# ── Diverging bar chart: sentiment shift per shared entity ────────
# Positive shift = more positive in Opt2 than Opt1; negative = more negative in Opt2
df_plot = cross_sent_df.dropna(subset=['Shift']).copy()
df_plot = df_plot.sort_values('Shift')

colors = ['#d32f2f' if s < 0 else '#388e3c' for s in df_plot['Shift']]

fig_cross_sent = go.Figure()
fig_cross_sent.add_trace(go.Bar(
    y=df_plot['Entity'], x=df_plot['Shift'],
    orientation='h', marker_color=colors,
    text=[f"{s:+.3f}" for s in df_plot['Shift']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Opt1 ratio: %{customdata[0]}<br>Opt2 ratio: %{customdata[1]}<br>Shift: %{x:+.3f}<extra></extra>',
    customdata=list(zip(df_plot['Opt1 Ratio'], df_plot['Opt2 Ratio'])),
))
fig_cross_sent.add_vline(x=0, line_dash='dash', line_color='grey')
fig_cross_sent.update_layout(
    title='Sentiment Shift: Option 2 vs Option 1 (shared entities)',
    xaxis_title='Sentiment Ratio Shift (Opt2 - Opt1)',
    yaxis_title='',
    height=max(400, len(df_plot) * 28),
    width=900,
    plot_bgcolor='white',
    xaxis=dict(zeroline=True),
    margin=dict(l=180),
)
fig_cross_sent.add_annotation(
    x=-0.5, y=1.05, xref='paper', yref='paper', showarrow=False,
    text='<-- More negative in Opt2', font=dict(color='#d32f2f', size=10))
fig_cross_sent.add_annotation(
    x=0.5, y=1.05, xref='paper', yref='paper', showarrow=False,
    text='More positive in Opt2 -->', font=dict(color='#388e3c', size=10))
fig_cross_sent.show()

# ── Directional Sentiment: Agency vs Patient ─────────────────────
print("\n\n" + "="*70)
print("  DIRECTIONAL SENTIMENT: AGENCY vs PATIENT")
print("="*70)
print("  Agency ratio:  based on out-degree (entity as SUBJECT)")
print("  Patient ratio: based on in-degree  (entity as OBJECT)")

directional_sentiment = {}
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    pos_sg = sentiment_analysis[label]['positive']['subgraph']
    neg_sg = sentiment_analysis[label]['negative']['subgraph']
    
    rows = []
    for v in g.vs:
        name = v['name']
        # Out-degree = entity as subject (agent)
        pos_out, neg_out = 0, 0
        # In-degree = entity as object (patient)
        pos_in, neg_in = 0, 0
        
        try:
            idx = pos_sg.vs.find(name=name).index
            pos_out = pos_sg.degree(idx, mode='out')
            pos_in = pos_sg.degree(idx, mode='in')
        except ValueError:
            pass
        try:
            idx = neg_sg.vs.find(name=name).index
            neg_out = neg_sg.degree(idx, mode='out')
            neg_in = neg_sg.degree(idx, mode='in')
        except ValueError:
            pass
        
        total_out = pos_out + neg_out
        total_in = pos_in + neg_in
        agency_ratio = (pos_out - neg_out) / total_out if total_out > 0 else 0.0
        patient_ratio = (pos_in - neg_in) / total_in if total_in > 0 else 0.0
        total_all = pos_out + neg_out + pos_in + neg_in
        
        if total_all > 0:
            rows.append({
                'Entity': name,
                'Pos Out': pos_out, 'Neg Out': neg_out, 'Agency Ratio': round(agency_ratio, 3),
                'Pos In': pos_in, 'Neg In': neg_in, 'Patient Ratio': round(patient_ratio, 3),
                'Total': total_all,
            })
    
    df_dir = pd.DataFrame(rows).sort_values('Total', ascending=False)
    directional_sentiment[label] = df_dir
    
    print(f"\n  {label} - Top 15 entities by directional sentiment:")
    print(df_dir.head(15).to_string(index=False))

# ── Cross-corpus directional comparison (shared entities) ─────────
print("\n\n" + "="*70)
print("  CROSS-CORPUS DIRECTIONAL SENTIMENT (shared entities)")
print("="*70)

dir_opt1 = directional_sentiment['Option 1'].set_index('Entity')
dir_opt2 = directional_sentiment['Option 2'].set_index('Entity')
shared_dir = set(dir_opt1.index) & set(dir_opt2.index)

cross_dir_rows = []
for entity in shared_dir:
    r1 = dir_opt1.loc[entity]
    r2 = dir_opt2.loc[entity]
    cross_dir_rows.append({
        'Entity': entity,
        'Opt1 Agency': r1['Agency Ratio'], 'Opt1 Patient': r1['Patient Ratio'],
        'Opt2 Agency': r2['Agency Ratio'], 'Opt2 Patient': r2['Patient Ratio'],
        'Agency Shift': round(r2['Agency Ratio'] - r1['Agency Ratio'], 3),
        'Patient Shift': round(r2['Patient Ratio'] - r1['Patient Ratio'], 3),
    })

cross_dir_df = pd.DataFrame(cross_dir_rows)
cross_dir_df['Max Shift'] = cross_dir_df[['Agency Shift', 'Patient Shift']].abs().max(axis=1)
cross_dir_df = cross_dir_df.sort_values('Max Shift', ascending=False).drop(columns='Max Shift')

print(f"\n  Shared entities with sentiment data: {len(cross_dir_df)}")
print("\n  Sorted by largest directional shift:")
print(cross_dir_df.to_string(index=False))

# ── Diverging bar chart: agency vs patient shift ──────────────────
df_plot_dir = cross_dir_df.copy()
df_plot_dir = df_plot_dir.sort_values('Agency Shift')

fig_dir_sent = make_subplots(rows=1, cols=2, shared_yaxes=True,
    subplot_titles=['Agency Shift (as Subject)', 'Patient Shift (as Object)'],
    horizontal_spacing=0.08)

colors_ag = ['#d32f2f' if s < 0 else '#388e3c' for s in df_plot_dir['Agency Shift']]
colors_pt = ['#d32f2f' if s < 0 else '#388e3c' for s in df_plot_dir['Patient Shift']]

fig_dir_sent.add_trace(go.Bar(
    y=df_plot_dir['Entity'], x=df_plot_dir['Agency Shift'],
    orientation='h', marker_color=colors_ag, name='Agency',
    showlegend=False,
), row=1, col=1)

fig_dir_sent.add_trace(go.Bar(
    y=df_plot_dir['Entity'], x=df_plot_dir['Patient Shift'],
    orientation='h', marker_color=colors_pt, name='Patient',
    showlegend=False,
), row=1, col=2)

fig_dir_sent.add_vline(x=0, line_dash='dash', line_color='grey', row=1, col=1)
fig_dir_sent.add_vline(x=0, line_dash='dash', line_color='grey', row=1, col=2)
fig_dir_sent.update_layout(
    title='Directional Sentiment Shift: Option 2 vs Option 1 (shared entities)',
    height=max(500, len(df_plot_dir) * 28), width=1100,
    plot_bgcolor='white', margin=dict(l=180),
)
fig_dir_sent.show()

  ANALYSIS 7: SENTIMENT-WEIGHTED NETWORK

  Option 1:
    Sentiment    Edges    % of total   Active nodes   Density
    ------------ -------- ------------ -------------- --------
    positive     86       14.0       % 114            0.0067
    negative     145      23.7       % 148            0.0067
    neutral      382      62.3       % 392            0.0025

  Option 2:
    Sentiment    Edges    % of total   Active nodes   Density
    ------------ -------- ------------ -------------- --------
    positive     32       11.6       % 50             0.0131
    negative     80       28.9       % 97             0.0086
    neutral      165      59.6       % 203            0.0040


Sentiment Centrality: Who is central in positive vs negative networks?
----------------------------------------------------------------------

  Option 1 - Top 15 entities by sentiment network participation:
           Entity  Positive_Degree  Negative_Degree  Sentiment_Ratio  Total
             Éire              



  CROSS-CORPUS SENTIMENT COMPARISON

  Shared entities with sentiment data: 33

  Cross-Corpus Sentiment Comparison (sorted by largest framing difference):
                  Entity  Opt1 Pos  Opt1 Neg  Opt1 Ratio  Opt2 Pos  Opt2 Neg  Opt2 Ratio  Shift
                    seat         1         0       1.000         0         1      -1.000 -2.000
                  Dublin         0         2      -1.000         1         0       1.000  2.000
                Security         0         1      -1.000         1         0       1.000  2.000
      British Government         1         0       1.000         1         3      -0.500 -1.500
               Sinn Féin         1         0       1.000         1         1       0.000 -1.000
                  Police         1         0       1.000         1         1       0.000 -1.000
                Catholic         2         2       0.000         0         2      -1.000 -1.000
             Nationalist         3         2       0.200         1        



  DIRECTIONAL SENTIMENT: AGENCY vs PATIENT
  Agency ratio:  based on out-degree (entity as SUBJECT)
  Patient ratio: based on in-degree  (entity as OBJECT)

  Option 1 - Top 15 entities by directional sentiment:
           Entity  Pos Out  Neg Out  Agency Ratio  Pos In  Neg In  Patient Ratio  Total
             Éire       10        5         0.333       2       4         -0.333     21
          Britain        7        4         0.273       3       5         -0.250     19
  Éamon de Valera        7        9        -0.125       0       2         -1.000     18
              IRA        2        9        -0.636       1       2         -0.333     14
          Belfast        0        7        -1.000       1       4         -0.600     12
         Unionist        1       10        -0.818       1       0          1.000     12
          Germany        2        8        -0.600       1       1          0.000     12
           Ulster        4        1         0.600       2       4         -0.333  

In [9]:
# Cell 8b: Structural Role Equivalence via Feature Profiles

print("="*70)
print("  ANALYSIS 7b: STRUCTURAL ROLE EQUIVALENCE")
print("="*70)
print("  Method: k-means clustering on network feature profiles")
print("  Entities with similar profiles play the same structural role")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

# ── Build feature profiles ────────────────────────────────────────
def build_role_profiles(g_d, g_u, sentiment_analysis_result, label):
    """Build a feature vector for each entity from pre-computed metrics."""
    pos_sg = sentiment_analysis_result['positive']['subgraph']
    neg_sg = sentiment_analysis_result['negative']['subgraph']
    
    hub = g_d.hub_score()
    auth = g_d.authority_score()
    btwn = g_d.betweenness()
    eigen = g_u.eigenvector_centrality()
    
    rows = []
    for i in range(g_d.vcount()):
        name = g_d.vs[i]['name']
        in_deg = g_d.degree(i, mode='in')
        out_deg = g_d.degree(i, mode='out')
        
        # Sentiment degrees
        pd_val = 0
        nd_val = 0
        try:
            idx_p = pos_sg.vs.find(name=name).index
            pd_val = pos_sg.degree(idx_p)
        except ValueError:
            pass
        try:
            idx_n = neg_sg.vs.find(name=name).index
            nd_val = neg_sg.degree(idx_n)
        except ValueError:
            pass
        total_pn = pd_val + nd_val
        sent_ratio = (pd_val - nd_val) / total_pn if total_pn > 0 else 0.0
        
        # Entity type encoding
        etype = g_d.vs[i]['type_broad'] if 'type_broad' in g_d.vs.attributes() else 'Entity'
        
        rows.append({
            'name': name,
            'type': etype,
            'in_degree': in_deg,
            'out_degree': out_deg,
            'hub': hub[i],
            'authority': auth[i],
            'betweenness': btwn[i],
            'eigenvector': eigen[i],
            'pos_degree': pd_val,
            'neg_degree': nd_val,
            'sentiment_ratio': sent_ratio,
        })
    
    return pd.DataFrame(rows)

# Build profiles for both corpora
profile_features = ['in_degree', 'out_degree', 'hub', 'authority',
                    'betweenness', 'eigenvector', 'pos_degree',
                    'neg_degree', 'sentiment_ratio']

role_profiles = {}
role_labels_all = {}

for label, g_d, g_u in [('Option 1', G_opt1, G_opt1_u), ('Option 2', G_opt2, G_opt2_u)]:
    df_prof = build_role_profiles(g_d, g_u, sentiment_analysis[label], label)
    role_profiles[label] = df_prof

# ── Determine optimal k via silhouette score ─────────────────────
from sklearn.metrics import silhouette_score

print("\n--- Silhouette analysis for role count ---")
for label in ['Option 1', 'Option 2']:
    df_prof = role_profiles[label]
    X = StandardScaler().fit_transform(df_prof[profile_features].fillna(0))
    
    sil_scores = {}
    for k in range(3, 9):
        km = KMeans(n_clusters=k, n_init=20, random_state=42)
        labs = km.fit_predict(X)
        sil = silhouette_score(X, labs)
        sil_scores[k] = sil
    
    best_k = max(sil_scores, key=sil_scores.get)
    print(f"\n  {label}:")
    for k, s in sil_scores.items():
        marker = ' <-- best' if k == best_k else ''
        print(f"    k={k}: silhouette={s:.3f}{marker}")
    
    # Use best k, but cap at 6 for interpretability
    use_k = min(best_k, 6)
    print(f"  Using k={use_k}")
    
    km_final = KMeans(n_clusters=use_k, n_init=20, random_state=42)
    X_scaled = StandardScaler().fit_transform(df_prof[profile_features].fillna(0))
    df_prof['role'] = km_final.fit_predict(X_scaled)
    role_profiles[label] = df_prof
    
    # Store role on graph vertices
    if label == 'Option 1':
        G_opt1.vs['role'] = df_prof['role'].tolist()
        G_opt1_u.vs['role'] = df_prof['role'].tolist()
    else:
        G_opt2.vs['role'] = df_prof['role'].tolist()
        G_opt2_u.vs['role'] = df_prof['role'].tolist()

# ── Role typology: characterize each role ─────────────────────────
print("\n\n--- Role Typology ---")

role_summaries = {}
for label in ['Option 1', 'Option 2']:
    df_prof = role_profiles[label]
    n_roles = df_prof['role'].nunique()
    
    print(f"\n  {label} ({n_roles} roles):")
    print(f"  {'Role':<6} {'n':<5} {'AvgDeg':<8} {'Hub':<8} {'Auth':<8} {'Btwn':<10} {'SentR':<8} {'Types':<25} {'Top Entities'}")
    print(f"  {'-'*6} {'-'*5} {'-'*8} {'-'*8} {'-'*8} {'-'*10} {'-'*8} {'-'*25} {'-'*30}")
    
    summaries = []
    for r in sorted(df_prof['role'].unique()):
        cluster = df_prof[df_prof['role'] == r]
        n = len(cluster)
        avg_deg = cluster['in_degree'].mean() + cluster['out_degree'].mean()
        avg_hub = cluster['hub'].mean()
        avg_auth = cluster['authority'].mean()
        avg_btwn = cluster['betweenness'].mean()
        avg_sent = cluster['sentiment_ratio'].mean()
        
        type_counts = Counter(cluster['type'])
        type_str = ', '.join(f"{t}:{c}" for t, c in type_counts.most_common(3))
        
        top3 = cluster.nlargest(3, 'eigenvector')['name'].tolist()
        top_str = ', '.join(top3)
        
        print(f"  {r:<6} {n:<5} {avg_deg:<8.1f} {avg_hub:<8.4f} {avg_auth:<8.4f} {avg_btwn:<10.1f} {avg_sent:<8.3f} {type_str:<25} {top_str}")
        
        summaries.append({
            'Role': r, 'n': n, 'Avg_Degree': round(avg_deg, 1),
            'Avg_Hub': round(avg_hub, 4), 'Avg_Authority': round(avg_auth, 4),
            'Avg_Betweenness': round(avg_btwn, 1),
            'Avg_Sentiment': round(avg_sent, 3), 'Types': type_str,
            'Top_Entities': top_str,
        })
    role_summaries[label] = pd.DataFrame(summaries)

# ── Cross-corpus role comparison (shared entities) ────────────────
print("\n\n--- Cross-Corpus Role Comparison (shared entities) ---")
print("-"*70)

opt1_prof = role_profiles['Option 1'].set_index('name')
opt2_prof = role_profiles['Option 2'].set_index('name')
shared_ents = set(opt1_prof.index) & set(opt2_prof.index)

if shared_ents:
    role_comparison_rows = []
    for entity in shared_ents:
        r1 = opt1_prof.loc[entity]
        r2 = opt2_prof.loc[entity]
        role_comparison_rows.append({
            'Entity': entity,
            'Opt1_Role': r1['role'],
            'Opt1_Hub': round(r1['hub'], 4),
            'Opt1_Auth': round(r1['authority'], 4),
            'Opt1_Sent': round(r1['sentiment_ratio'], 3),
            'Opt2_Role': r2['role'],
            'Opt2_Hub': round(r2['hub'], 4),
            'Opt2_Auth': round(r2['authority'], 4),
            'Opt2_Sent': round(r2['sentiment_ratio'], 3),
            'Role_Change': 'Yes' if r1['role'] != r2['role'] else '',
        })
    
    role_comp_df = pd.DataFrame(role_comparison_rows).sort_values(
        ['Role_Change', 'Opt1_Role'], ascending=[False, True])
    
    n_changed = sum(1 for r in role_comparison_rows if r['Role_Change'] == 'Yes')
    print(f"\n  Shared entities: {len(shared_ents)}")
    print(f"  Role changes: {n_changed} ({100*n_changed/len(shared_ents):.0f}%)")
    print(f"\n  Entities with role changes:")
    changed = role_comp_df[role_comp_df['Role_Change'] == 'Yes']
    if not changed.empty:
        print(changed.to_string(index=False))
    else:
        print("  (no role changes detected)")

# ── Visualisation: feature profile heatmap per role ───────────────
for label in ['Option 1', 'Option 2']:
    df_prof = role_profiles[label]
    n_roles = df_prof['role'].nunique()
    
    # Mean feature values by role (normalized)
    X = StandardScaler().fit_transform(df_prof[profile_features].fillna(0))
    df_norm = pd.DataFrame(X, columns=profile_features)
    df_norm['role'] = df_prof['role'].values
    role_means = df_norm.groupby('role')[profile_features].mean()
    
    fig_heat = go.Figure(data=go.Heatmap(
        z=role_means.values,
        x=[f.replace('_', ' ').title() for f in profile_features],
        y=[f'Role {r}' for r in role_means.index],
        colorscale='RdBu_r', zmid=0,
        text=np.round(role_means.values, 2), texttemplate='%{text}',
    ))
    fig_heat.update_layout(
        title=f'{label} — Structural Role Feature Profiles (standardized)',
        width=900, height=300 + n_roles * 40,
        xaxis_title='Feature', yaxis_title='Role',
    )
    fig_heat.show()

# ── Network colored by role ──────────────────────────────────────
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    fig_role = plot_network_plotly(g, layout_algo='fr',
        title=f'{label} - Structural Roles', color_attr='role')
    fig_role.show()


# ── List all entities in non-peripheral roles ─────────────────────
print("\n\n" + "="*70)
print("  NON-PERIPHERAL ROLE MEMBERSHIP (Roles 1+)")
print("="*70)

for label in ['Option 1', 'Option 2']:
    df_prof = role_profiles[label]
    peripheral_role = df_prof['role'].value_counts().idxmax()  # largest cluster = periphery
    
    non_peripheral = df_prof[df_prof['role'] != peripheral_role].copy()
    non_peripheral = non_peripheral.sort_values(['role', 'eigenvector'], ascending=[True, False])
    
    print(f"\n  {label} (peripheral = Role {peripheral_role}, n={len(df_prof[df_prof['role']==peripheral_role])})")
    print(f"  Non-peripheral entities: {len(non_peripheral)}")
    
    for role in sorted(non_peripheral['role'].unique()):
        subset = non_peripheral[non_peripheral['role'] == role]
        avg_hub = subset['hub'].mean()
        avg_auth = subset['authority'].mean()
        avg_btwn = subset['betweenness'].mean()
        avg_sent = subset['sentiment_ratio'].mean()
        
        # Assign descriptive label based on profile
        if avg_hub > 0.3 or avg_btwn > 1000:
            role_desc = "Narrative Protagonists"
        elif avg_hub > 0.05 or avg_btwn > 100:
            role_desc = "Secondary Actors"
        else:
            role_desc = "Supporting Cast"
        
        print(f"\n  --- Role {role}: {role_desc} (n={len(subset)}) ---")
        print(f"      Profile: Hub={avg_hub:.4f}, Auth={avg_auth:.4f}, Btwn={avg_btwn:.1f}, Sent={avg_sent:.3f}")
        print(f"      {'Entity':<35} {'Type':<10} {'Degree':<8} {'Hub':<8} {'Auth':<8} {'Btwn':<10} {'SentR':<8}")
        print(f"      {'-'*35} {'-'*10} {'-'*8} {'-'*8} {'-'*8} {'-'*10} {'-'*8}")
        
        for _, row in subset.iterrows():
            deg = row['in_degree'] + row['out_degree']
            nm = row['name']
            tp = row['type']
            hb = row['hub']
            au = row['authority']
            bt = row['betweenness']
            sr = row['sentiment_ratio']
            print(f"      {nm:<35} {tp:<10} {deg:<8.0f} {hb:<8.4f} {au:<8.4f} {bt:<10.1f} {sr:<8.3f}")


  ANALYSIS 7b: STRUCTURAL ROLE EQUIVALENCE
  Method: k-means clustering on network feature profiles
  Entities with similar profiles play the same structural role

--- Silhouette analysis for role count ---

  Option 1:
    k=3: silhouette=0.770 <-- best
    k=4: silhouette=0.440
    k=5: silhouette=0.473
    k=6: silhouette=0.545
    k=7: silhouette=0.545
    k=8: silhouette=0.553
  Using k=3

  Option 2:
    k=3: silhouette=0.613 <-- best
    k=4: silhouette=0.513
    k=5: silhouette=0.417
    k=6: silhouette=0.470
    k=7: silhouette=0.475
    k=8: silhouette=0.479
  Using k=3


--- Role Typology ---

  Option 1 (3 roles):
  Role   n     AvgDeg   Hub      Auth     Btwn       SentR    Types                     Top Entities
  ------ ----- -------- -------- -------- ---------- -------- ------------------------- ------------------------------
  0      511   1.5      0.0026   0.0582   23.7       -0.064   Entity:413, Event:72, Figure:26 Treaty Port, France, Emergency
  1      17    17.7  



  NON-PERIPHERAL ROLE MEMBERSHIP (Roles 1+)

  Option 1 (peripheral = Role 0, n=511)
  Non-peripheral entities: 20

  --- Role 1: Narrative Protagonists (n=17) ---
      Profile: Hub=0.1058, Auth=0.4623, Btwn=2502.7, Sent=-0.340
      Entity                              Type       Degree   Hub      Auth     Btwn       SentR   
      ----------------------------------- ---------- -------- -------- -------- ---------- --------
      Irish Free State                    Entity     28       0.3082   0.7270   5009.6     -0.500  
      World War II                        Event      21       0.0355   0.9449   2832.1     0.333   
      Ulster                              Entity     25       0.1664   0.2830   2802.0     0.091   
      Anglo-Irish Treaty                  Event      7        0.0596   0.7419   448.0      1.000   
      Neutrality                          Event      10       0.0802   0.7667   798.9      0.000   
      Winston Churchill                   Figure     21       0.1825 

In [13]:
# ── Statistical Significance: Shared Entity Sentiment ─────────────
from scipy import stats
from statsmodels.stats.multitest import multipletests

print("\n\n" + "="*70)
print("  STATISTICAL SIGNIFICANCE: SHARED ENTITY SENTIMENT")
print("="*70)

# Use the shared entities from directional sentiment
dir_opt1 = directional_sentiment['Option 1'].set_index('Entity')
dir_opt2 = directional_sentiment['Option 2'].set_index('Entity')
shared_dir = set(dir_opt1.index) & set(dir_opt2.index)

# ── Test 1: Per-entity Fisher's exact test ────────────────────────
print("\n--- Test 1: Per-Entity Fisher's Exact Test ---")
print("  H0: For each shared entity, the ratio of positive to negative")
print("      edges is the same across both corpora.")
print("  Odds ratio uses Haldane-Anscombe correction (+0.5) to handle zeros.\n")

fisher_rows = []
for entity in shared_dir:
    r1 = dir_opt1.loc[entity]
    r2 = dir_opt2.loc[entity]
    
    p1 = int(r1['Pos Out'] + r1['Pos In'])
    n1 = int(r1['Neg Out'] + r1['Neg In'])
    p2 = int(r2['Pos Out'] + r2['Pos In'])
    n2 = int(r2['Neg Out'] + r2['Neg In'])
    
    if p1 + n1 + p2 + n2 == 0:
        continue
    
    table = [[p1, n1], [p2, n2]]
    _, p_val = stats.fisher_exact(table)
    
    # Haldane-Anscombe corrected odds ratio
    or_corrected = ((p1 + 0.5) * (n2 + 0.5)) / ((n1 + 0.5) * (p2 + 0.5))
    
    fisher_rows.append({
        'Entity': entity,
        'Opt1 Pos': p1, 'Opt1 Neg': n1,
        'Opt2 Pos': p2, 'Opt2 Neg': n2,
        'Odds Ratio': round(or_corrected, 3),
        'p-value': round(p_val, 4),
        'Sig': '*' if p_val < 0.05 else '',
    })

fisher_df = pd.DataFrame(fisher_rows).sort_values('p-value')

# FDR correction
if len(fisher_df) > 1:
    reject, p_adj, _, _ = multipletests(fisher_df['p-value'], method='fdr_bh')
    fisher_df['p-adjusted'] = [round(p, 4) for p in p_adj]
    fisher_df['FDR Sig'] = ['*' if r else '' for r in reject]

print(f"  Entities tested: {len(fisher_df)}")
print(f"  Significant at p < 0.05: {sum(1 for _, r in fisher_df.iterrows() if r['Sig'] == '*')}")
print(f"  Significant after FDR correction: {sum(1 for _, r in fisher_df.iterrows() if r['FDR Sig'] == '*')}")
print(f"\n{fisher_df.to_string(index=False)}")

# ── Test 2: Wilcoxon on directional sentiment shifts ──────────────
print("\n\n--- Test 2: Wilcoxon Signed-Rank Test on Sentiment Shifts ---")
print("  H0: The median shift across shared entities is zero.\n")

for shift_col, shift_name in [('Agency Shift', 'Agency'), ('Patient Shift', 'Patient')]:
    vals = cross_dir_df[shift_col].dropna().values
    # Remove exact zeros (Wilcoxon discards them anyway)
    vals_nz = vals[vals != 0]
    if len(vals_nz) >= 10:
        stat_d, p_d = stats.wilcoxon(vals_nz)
        median_d = np.median(vals)
        n = len(vals_nz)
        r_rb = 1 - (2 * stat_d) / (n * (n + 1) / 2)
        effect = 'negligible' if abs(r_rb) < 0.1 else 'small' if abs(r_rb) < 0.3 else 'medium' if abs(r_rb) < 0.5 else 'large'
        print(f"  {shift_name} Shift:")
        print(f"    n = {len(vals)} ({len(vals_nz)} non-zero), median = {median_d:.3f}")
        print(f"    W = {stat_d:.1f}, p = {p_d:.4f} ({'Significant' if p_d < 0.05 else 'Not significant'})")
        print(f"    Effect size (rank-biserial r) = {r_rb:.3f} ({effect})")
    else:
        print(f"  {shift_name}: n = {len(vals)} ({len(vals_nz)} non-zero), too few non-zero for Wilcoxon (need >= 10)")

# ── Test 3: Combined sentiment ratio shift ────────────────────────
print(f"\n  Combined Sentiment Ratio Shift:")
shifts_combined = cross_sent_df['Shift'].dropna().values
shifts_nz = shifts_combined[shifts_combined != 0]
if len(shifts_nz) >= 10:
    stat_w, p_w = stats.wilcoxon(shifts_nz)
    median_shift = np.median(shifts_combined)
    n = len(shifts_nz)
    r_rb = 1 - (2 * stat_w) / (n * (n + 1) / 2)
    effect = 'negligible' if abs(r_rb) < 0.1 else 'small' if abs(r_rb) < 0.3 else 'medium' if abs(r_rb) < 0.5 else 'large'
    print(f"    n = {len(shifts_combined)} ({n} non-zero), median = {median_shift:.3f}")
    print(f"    W = {stat_w:.1f}, p = {p_w:.4f} ({'Significant' if p_w < 0.05 else 'Not significant'})")
    print(f"    Effect size (rank-biserial r) = {r_rb:.3f} ({effect})")
else:
    print(f"    n = {len(shifts_combined)} ({len(shifts_nz)} non-zero), too few non-zero for Wilcoxon (need >= 10)")

# ── Test 4: Per-entity Permutation Test on Sentiment Shift ────────
print("\n--- Test 4: Per-Entity Permutation Test on Sentiment Shift ---")
print("  H0: For each shared entity, the sentiment shift between corpora")
print("      is no larger than expected by random assignment of edges.")
print("  Method: 10,000 permutations of pooled pos/neg edges.\n")

np.random.seed(42)
N_PERMS = 10000

def compute_ratio(pos, neg):
    total = pos + neg
    return (pos - neg) / total if total > 0 else 0.0

perm_rows = []
for entity in shared_dir:
    r1 = dir_opt1.loc[entity]
    r2 = dir_opt2.loc[entity]
    
    p1 = int(r1['Pos Out'] + r1['Pos In'])
    n1 = int(r1['Neg Out'] + r1['Neg In'])
    p2 = int(r2['Pos Out'] + r2['Pos In'])
    n2 = int(r2['Neg Out'] + r2['Neg In'])
    
    total_edges = p1 + n1 + p2 + n2
    if total_edges == 0:
        continue
    
    # Observed shift
    ratio1 = compute_ratio(p1, n1)
    ratio2 = compute_ratio(p2, n2)
    observed_shift = ratio2 - ratio1
    
    # Pool all edges: 1 = positive, 0 = negative
    pooled = [1] * (p1 + p2) + [0] * (n1 + n2)
    size1 = p1 + n1  # number of edges in corpus 1
    
    # Permutation
    null_shifts = np.zeros(N_PERMS)
    pooled_arr = np.array(pooled)
    for perm in range(N_PERMS):
        np.random.shuffle(pooled_arr)
        perm_p1 = pooled_arr[:size1].sum()
        perm_n1 = size1 - perm_p1
        perm_p2 = pooled_arr[size1:].sum()
        perm_n2 = len(pooled_arr) - size1 - perm_p2
        r1_perm = compute_ratio(perm_p1, perm_n1)
        r2_perm = compute_ratio(perm_p2, perm_n2)
        null_shifts[perm] = r2_perm - r1_perm
    
    # Two-tailed p-value
    p_val = np.mean(np.abs(null_shifts) >= np.abs(observed_shift))
    
    perm_rows.append({
        'Entity': entity,
        'Opt1 Pos': p1, 'Opt1 Neg': n1, 'Opt1 Ratio': round(ratio1, 3),
        'Opt2 Pos': p2, 'Opt2 Neg': n2, 'Opt2 Ratio': round(ratio2, 3),
        'Shift': round(observed_shift, 3),
        'p-value': round(p_val, 4),
        'Sig': '*' if p_val < 0.05 else '',
    })

perm_df = pd.DataFrame(perm_rows).sort_values('p-value')

# FDR correction
if len(perm_df) > 1:
    reject, p_adj, _, _ = multipletests(perm_df['p-value'], method='fdr_bh')
    perm_df['p-adjusted'] = [round(p, 4) for p in p_adj]
    perm_df['FDR Sig'] = ['*' if r else '' for r in reject]

print(f"  Entities tested: {len(perm_df)}")
print(f"  Significant at p < 0.05: {sum(1 for _, r in perm_df.iterrows() if r['Sig'] == '*')}")
print(f"  Significant after FDR correction: {sum(1 for _, r in perm_df.iterrows() if r['FDR Sig'] == '*')}")
print(f"\n{perm_df.to_string(index=False)}")



  STATISTICAL SIGNIFICANCE: SHARED ENTITY SENTIMENT

--- Test 1: Per-Entity Fisher's Exact Test ---
  H0: For each shared entity, the ratio of positive to negative
      edges is the same across both corpora.
  Odds ratio uses Haldane-Anscombe correction (+0.5) to handle zeros.

  Entities tested: 16
  Significant at p < 0.05: 0
  Significant after FDR correction: 0

             Entity  Opt1 Pos  Opt1 Neg  Opt2 Pos  Opt2 Neg  Odds Ratio  p-value Sig  p-adjusted FDR Sig
        Nationalist         3         2         1         5       5.133   0.2424             1.0        
             Dublin         0         2         1         0       0.067   0.3333             1.0        
 British Government         1         0         1         3       7.000   0.4000             1.0        
           Catholic         2         2         0         2       5.000   0.4667             1.0        
         Government         5         2         2         2       2.200   0.5758             1.0       

In [ ]:
# Cell 9: Cross-Corpus Synthesis

print("="*70)
print("  ANALYSIS 8: CROSS-CORPUS SYNTHESIS")
print("="*70)

# ── Narrative structure fingerprint ────────────────────────────────
print("\nNarrative Structure Fingerprint")
print("-"*60)

fingerprint = []
for label, g, g_u in [('Option 1', G_opt1, G_opt1_u), ('Option 2', G_opt2, G_opt2_u)]:
    comps = g_u.connected_components()
    giant = max(len(c) for c in comps)
    sa = sentiment_analysis[label]
    reach = reach_opt1 if label == 'Option 1' else reach_opt2
    fingerprint.append({
        'Metric': label, 'Nodes': g.vcount(), 'Edges': g.ecount(),
        'Density': round(g.density(), 4),
        'Components': len(comps),
        'Giant Component %': round(100 * giant / g.vcount(), 1),
        'Reachability @20': round(reach[max_steps], 3),
        'Modularity': round(community_results[label]['modularity'], 4),
        'N Communities': community_results[label]['n_communities'],
        'Positive Edges %': round(100 * sa['positive']['edges'] / g.ecount(), 1),
        'Negative Edges %': round(100 * sa['negative']['edges'] / g.ecount(), 1),
    })

fp_df = pd.DataFrame(fingerprint).T
fp_df.columns = fp_df.iloc[0]
fp_df = fp_df.iloc[1:]
print(fp_df.to_string())

# ── Master comparison table ────────────────────────────────────────
print("\n\nShared Entity Master Comparison")
print("-"*70)

opt1_set = set(G_opt1.vs['name'])
opt2_set = set(G_opt2.vs['name'])
shared = opt1_set & opt2_set

master_rows = []
for entity in shared:
    row = {'Entity': entity}
    for pfx, label in [('Opt1', 'Option 1'), ('Opt2', 'Option 2')]:
        g = G_opt1 if pfx == 'Opt1' else G_opt2
        idx = g.vs['name'].index(entity)
        cent = centrality_results[label]
        hits = hits_results[label]
        ni = list(cent['names']).index(entity)
        row[f'{pfx}_Degree'] = round(cent['total_degree'][ni], 4)
        row[f'{pfx}_Betweenness'] = round(cent['betweenness'][ni], 4)
        row[f'{pfx}_Eigenvector'] = round(cent['eigenvector'][ni], 4)
        row[f'{pfx}_Hub'] = round(hits['hub'][ni], 4)
        row[f'{pfx}_Authority'] = round(hits['authority'][ni], 4)
        row[f'{pfx}_Community'] = g.vs[idx]['community']
    master_rows.append(row)

master_df = pd.DataFrame(master_rows)
master_df['Avg_Eigen'] = (master_df['Opt1_Eigenvector'] + master_df['Opt2_Eigenvector']) / 2
master_df = master_df.sort_values('Avg_Eigen', ascending=False)

print(f"\n  Shared entities: {len(master_df)}")
print(f"\n  Top 20 (by average eigenvector centrality):")
cols = ['Entity', 'Opt1_Eigenvector', 'Opt2_Eigenvector',
        'Opt1_Hub', 'Opt2_Hub', 'Opt1_Authority', 'Opt2_Authority',
        'Opt1_Community', 'Opt2_Community']
print(master_df.head(20)[cols].to_string(index=False))

# ── Export ─────────────────────────────────────────────────────────
os.makedirs('../graph_data', exist_ok=True)
master_df.to_csv('../graph_data/shared_entity_master_comparison.csv', index=False)
fp_df.to_csv('../graph_data/narrative_structure_fingerprint.csv')
print("\nExported: shared_entity_master_comparison.csv")
print("Exported: narrative_structure_fingerprint.csv")


In [ ]:
# Cell 10: Export & Full Network Visualizations

print("="*70)
print("  EXPORTS & NETWORK VISUALIZATIONS")
print("="*70)

# ── Full network visualizations ────────────────────────────────────
fig1 = plot_network_plotly(G_opt1, layout_algo='fr',
    title='Option 1 (Protestant/Unionist) - Entity Types',
    color_attr='type_broad')
fig1.show()

fig2 = plot_network_plotly(G_opt2, layout_algo='fr',
    title='Option 2 (Catholic/Nationalist) - Entity Types',
    color_attr='type_broad')
fig2.show()

# ── Export GraphML for Gephi ───────────────────────────────────────
os.makedirs('../graph_data', exist_ok=True)
G_opt1.write_graphml('../graph_data/option1_network.graphml')
G_opt2.write_graphml('../graph_data/option2_network.graphml')
G_combined.write_graphml('../graph_data/combined_network.graphml')
print("Exported: option1_network.graphml")
print("Exported: option2_network.graphml")
print("Exported: combined_network.graphml")
print("   (Open in Gephi for advanced visualization)")

# ── Export centrality data ─────────────────────────────────────────
for label in ['Option 1', 'Option 2']:
    res = centrality_results[label]
    df_export = pd.DataFrame({
        'Entity': res['names'], 'Type': res['types'],
        'Degree': res['total_degree'], 'In_Degree': res['in_degree'],
        'Out_Degree': res['out_degree'], 'Betweenness': res['betweenness'],
        'Eigenvector': res['eigenvector'],
        'Hub': hits_results[label]['hub'], 'Authority': hits_results[label]['authority'],
    })
    fname = f"../graph_data/{label.lower().replace(' ', '_')}_centrality.csv"
    df_export.to_csv(fname, index=False)
    print(f"Exported: {fname}")

print("\nAll analyses complete!")


In [ ]:
# Cell 11: Export All Results for PI Report
# Saves: figures as PNG (300 DPI), tables as LaTeX .tex files
# Requires: pip install kaleido (for Plotly static export)

import os
from sklearn.preprocessing import StandardScaler

REPORT_DIR = '../graph_data/pi_report'
FIG_DIR = os.path.join(REPORT_DIR, 'figures')
TAB_DIR = os.path.join(REPORT_DIR, 'tables')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

def save_fig(fig, name, width=1000, height=600, scale=3):
    """Save Plotly figure as PNG"""
    path = os.path.join(FIG_DIR, f'{name}.png')
    fig.write_image(path, width=width, height=height, scale=scale)
    print(f'  Saved: figures/{name}.png')

def save_latex(df, name, caption, label, **kwargs):
    """Save DataFrame as LaTeX table"""
    path = os.path.join(TAB_DIR, f'{name}.tex')
    latex = df.to_latex(
        index=kwargs.get('index', False),
        float_format=kwargs.get('float_format', '%.4f'),
        caption=caption,
        label=f'tab:{label}',
        position='htbp',
        column_format=kwargs.get('column_format', None),
    )
    with open(path, 'w') as f:
        f.write(latex)
    print(f'  Saved: tables/{name}.tex')

print('='*70)
print('  EXPORTING ALL RESULTS FOR PI REPORT')
print('='*70)
print(f'  Output: {os.path.abspath(REPORT_DIR)}')

# ═══════════════════════════════════════════════════════════════════
# ANALYSIS 1: Descriptive Characteristics
# ═══════════════════════════════════════════════════════════════════
print('\n[1/8] Descriptive Characteristics')

# Table 1: Node type distribution
save_latex(table1, 'table1_node_types',
    caption='Node Type Distribution by Corpus (cf.\\ Bearman Table 1)',
    label='node_types')

# Table 2: Tie type cross-tabulation
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    tie_rows = []
    for e in g.es:
        src_t = g.vs[e.source]['type_broad']
        tgt_t = g.vs[e.target]['type_broad']
        tie_rows.append(f'{src_t} -> {tgt_t}')
    tie_counts = Counter(tie_rows)
    total = sum(tie_counts.values())
    tie_df = pd.DataFrame([
        {'Tie Type': k, 'n': v, 'Pct': round(100*v/total, 1)}
        for k, v in sorted(tie_counts.items(), key=lambda x: -x[1])
    ])
    tag = label.lower().replace(' ', '')
    save_latex(tie_df, f'table2_tie_types_{tag}',
        caption=f'Tie Type Cross-Tabulation — {label} (cf.\\ Bearman Table 2)',
        label=f'tie_types_{tag}',
        float_format='%.1f')

# Table: Basic network metrics
metrics_rows = []
for lbl, g in [('Option 1', G_opt1), ('Option 2', G_opt2), ('Combined', G_combined)]:
    degrees = g.degree()
    metrics_rows.append({
        'Corpus': lbl, 'Nodes': g.vcount(), 'Edges': g.ecount(),
        'Density': round(g.density(), 4), 'Avg Degree': round(np.mean(degrees), 2),
        'Max Degree': max(degrees), 'Reciprocity': round(g.reciprocity(), 3),
    })
metrics_df = pd.DataFrame(metrics_rows)
save_latex(metrics_df, 'table_basic_metrics',
    caption='Basic Network Metrics Comparison',
    label='basic_metrics', float_format='%.4f')

# Figures
save_fig(fig_types, 'fig1_node_type_distribution', width=900, height=450)
save_fig(fig_heat, 'fig2_tie_type_heatmap', width=900, height=450)

# ═══════════════════════════════════════════════════════════════════
# ANALYSIS 2: Component Analysis
# ═══════════════════════════════════════════════════════════════════
print('\n[2/8] Component Analysis')

comp_rows = []
for label, g_u in [('Option 1', G_opt1_u), ('Option 2', G_opt2_u)]:
    comps = g_u.connected_components()
    sizes = [len(c) for c in comps]
    giant = max(sizes)
    total = g_u.vcount()
    comp_rows.append({
        'Corpus': label, 'Components': len(sizes),
        'Giant Component': giant,
        'Giant Pct': round(100*giant/total, 1),
        'Isolates': sizes.count(1),
        'Total Nodes': total,
    })
comp_summary = pd.DataFrame(comp_rows)
save_latex(comp_summary, 'table3_components',
    caption='Component Analysis Summary (cf.\\ Bearman Fig.\\ 3)',
    label='components', float_format='%.1f')

save_fig(fig_comp, 'fig3_component_distribution', width=1000, height=450)

# ═══════════════════════════════════════════════════════════════════
# ANALYSIS 3: Centrality
# ═══════════════════════════════════════════════════════════════════
print('\n[3/8] Centrality Analysis')

# Mean centrality by entity type (Bearman Table 3)
for label in ['Option 1', 'Option 2']:
    res = centrality_results[label]
    df_cent = pd.DataFrame({
        'Type': res['types'], 'Degree': res['total_degree'],
        'Betweenness': res['betweenness'], 'Eigenvector': res['eigenvector'],
    })
    grouped = df_cent.groupby('Type').agg(
        n=('Type', 'count'), Mean_Degree=('Degree', 'mean'),
        Mean_Betweenness=('Betweenness', 'mean'),
        Mean_Eigenvector=('Eigenvector', 'mean'),
    ).round(4)
    tag = label.lower().replace(' ', '')
    save_latex(grouped, f'table4_centrality_by_type_{tag}',
        caption=f'Mean Centrality by Entity Type — {label} (cf.\\ Bearman Table 3)',
        label=f'centrality_type_{tag}', index=True)

# Relative centrality matrix
for label in ['Option 1', 'Option 2']:
    res = centrality_results[label]
    df_c = pd.DataFrame({'type': res['types'], 'eigenvector': res['eigenvector']})
    means = df_c.groupby('type')['eigenvector'].mean()
    types_p = sorted(means.index.tolist())
    matrix = pd.DataFrame(index=types_p, columns=types_p, dtype=float)
    for t1 in types_p:
        for t2 in types_p:
            matrix.loc[t1, t2] = round(means[t1], 4) if t1 == t2 else (
                round(means[t1]/means[t2], 2) if means[t2] > 0 else np.inf)
    tag = label.lower().replace(' ', '')
    save_latex(matrix, f'table4b_relative_centrality_{tag}',
        caption=f'Relative Centrality Matrix — {label} (diagonal = mean eigenvector)',
        label=f'rel_centrality_{tag}', index=True, float_format='%.3f')

# Top 10 entities
for label in ['Option 1', 'Option 2']:
    res = centrality_results[label]
    df_top = pd.DataFrame({
        'Entity': res['names'], 'Type': res['types'],
        'Degree': res['total_degree'], 'Betweenness': res['betweenness'],
        'Eigenvector': res['eigenvector'],
    }).sort_values('Eigenvector', ascending=False).head(10)
    tag = label.lower().replace(' ', '')
    save_latex(df_top, f'table4c_top10_centrality_{tag}',
        caption=f'Top 10 Most Central Entities — {label}',
        label=f'top10_centrality_{tag}')

save_fig(fig_cent, 'fig4_top10_eigenvector', width=1100, height=550)

# ═══════════════════════════════════════════════════════════════════
# ANALYSIS 4: Reachability
# ═══════════════════════════════════════════════════════════════════
print('\n[4/8] Reachability')

reach_df = pd.DataFrame({
    'Steps': list(range(1, max_steps+1)),
    'Option 1': [round(reach_opt1[s], 4) for s in range(1, max_steps+1)],
    'Option 2': [round(reach_opt2[s], 4) for s in range(1, max_steps+1)],
})
save_latex(reach_df, 'table5_reachability',
    caption='Mean Fraction of Elements Reached at $n$ Steps (cf.\\ Bearman Fig.\\ 5)',
    label='reachability')

save_fig(fig_reach, 'fig5_reachability_curves', width=900, height=550)

# ═══════════════════════════════════════════════════════════════════
# ANALYSIS 5: HITS
# ═══════════════════════════════════════════════════════════════════
print('\n[5/8] HITS Hub & Authority')

for label in ['Option 1', 'Option 2']:
    res = hits_results[label]
    df_h = pd.DataFrame({
        'Entity': res['names'], 'Type': res['types'],
        'Hub': res['hub'], 'Authority': res['authority'],
    })
    tag = label.lower().replace(' ', '')
    # Top hubs
    top_hubs = df_h.sort_values('Hub', ascending=False).head(10)
    save_latex(top_hubs, f'table6a_top_hubs_{tag}',
        caption=f'Top 10 Hubs (Narrative Drivers) — {label}',
        label=f'top_hubs_{tag}')
    # Top authorities
    top_auth = df_h.sort_values('Authority', ascending=False).head(10)
    save_latex(top_auth, f'table6b_top_authorities_{tag}',
        caption=f'Top 10 Authorities (Narrative Focal Points) — {label}',
        label=f'top_auth_{tag}')

# Cross-corpus comparison
opt1_names = set(hits_results['Option 1']['names'])
opt2_names = set(hits_results['Option 2']['names'])
shared_ents = opt1_names & opt2_names

if shared_ents:
    rows = []
    for entity in shared_ents:
        idx1 = list(hits_results['Option 1']['names']).index(entity)
        idx2 = list(hits_results['Option 2']['names']).index(entity)
        h1 = hits_results['Option 1']['hub'][idx1]
        a1 = hits_results['Option 1']['authority'][idx1]
        h2 = hits_results['Option 2']['hub'][idx2]
        a2 = hits_results['Option 2']['authority'][idx2]
        role1 = 'Hub' if h1 > a1 else 'Authority'
        role2 = 'Hub' if h2 > a2 else 'Authority'
        rows.append({
            'Entity': entity,
            'Opt1 Hub': round(h1, 4), 'Opt1 Auth': round(a1, 4), 'Opt1 Role': role1,
            'Opt2 Hub': round(h2, 4), 'Opt2 Auth': round(a2, 4), 'Opt2 Role': role2,
            'Role Change': 'Yes' if role1 != role2 else '',
        })
    hits_shared_df = pd.DataFrame(rows).sort_values('Opt1 Hub', ascending=False)
    save_latex(hits_shared_df, 'table6c_hits_shared_entities',
        caption='HITS Comparison for Shared Entities (Hub vs Authority Role Across Corpora)',
        label='hits_shared')

    # Role reversals only
    reversals = hits_shared_df[hits_shared_df['Role Change'] == 'Yes']
    if not reversals.empty:
        save_latex(reversals, 'table6d_hits_role_reversals',
            caption='Entities with Hub/Authority Role Reversal Across Corpora',
            label='hits_reversals')

save_fig(fig_hits, 'fig6_hits_scatter', width=1100, height=550)

# HITS network visualizations (Cell 6b)
def plot_hits_network(g, scores, score_type, title, top_n=8, width=900, height=700):
    scores = np.array(scores, dtype=float)
    layout = g.layout('fr')
    coords = np.array(layout.coords)
    sizes = 4 + 31 * (scores / (scores.max() + 1e-9))
    if score_type == 'Hub':
        rgb_high = (232, 145, 58)
    else:
        rgb_high = (74, 144, 217)
    rgb_low = (230, 230, 230)
    node_colors = []
    for s in scores / (scores.max() + 1e-9):
        r = int(rgb_low[0] + s * (rgb_high[0] - rgb_low[0]))
        gv = int(rgb_low[1] + s * (rgb_high[1] - rgb_low[1]))
        b = int(rgb_low[2] + s * (rgb_high[2] - rgb_low[2]))
        node_colors.append(f'rgb({r},{gv},{b})')
    edge_x, edge_y = [], []
    for e in g.es:
        x0, y0 = coords[e.source]
        x1, y1 = coords[e.target]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
    edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.3, color='#dddddd'), hoverinfo='none')
    node_trace = go.Scatter(
        x=coords[:, 0].tolist(), y=coords[:, 1].tolist(),
        mode='markers', showlegend=False,
        marker=dict(size=sizes.tolist(), color=node_colors,
                    line=dict(width=0.5, color='white')),
        text=[f"{g.vs[i]['name']}<br>{score_type}: {scores[i]:.4f}" for i in range(g.vcount())],
        hovertemplate='<b>%{text}</b><extra></extra>')
    top_indices = np.argsort(scores)[-top_n:]
    label_trace = go.Scatter(
        x=coords[top_indices, 0].tolist(), y=coords[top_indices, 1].tolist(),
        mode='text', showlegend=False,
        text=[g.vs[i]['name'] for i in top_indices],
        textposition='top center', textfont=dict(size=10, color='black'))
    fig = go.Figure(data=[edge_trace, node_trace, label_trace],
        layout=go.Layout(title=title, width=width, height=height,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor='white', hovermode='closest'))
    return fig

for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    hub = g.hub_score()
    auth = g.authority_score()
    tag = label.lower().replace(' ', '')
    fig_h = plot_hits_network(g, hub, 'Hub',
        title=f'{label} — Hubs (narrative drivers)')
    save_fig(fig_h, f'fig6b_hits_hubs_{tag}', width=1000, height=750)
    fig_a = plot_hits_network(g, auth, 'Authority',
        title=f'{label} — Authorities (narrative focal points)')
    save_fig(fig_a, f'fig6c_hits_authorities_{tag}', width=1000, height=750)

# ═══════════════════════════════════════════════════════════════════
# ANALYSIS 6: Community Detection
# ═══════════════════════════════════════════════════════════════════
print('\n[6/8] Community Detection')

for label, g_u, g_d in [('Option 1', G_opt1_u, G_opt1), ('Option 2', G_opt2_u, G_opt2)]:
    membership = community_results[label]['membership']
    n_communities = community_results[label]['n_communities']
    tag = label.lower().replace(' ', '')

    # Cluster summary table
    cluster_rows = []
    for c in range(n_communities):
        members = [i for i, m in enumerate(membership) if m == c]
        size = len(members)
        if size < 2:
            continue
        type_counts = Counter(g_u.vs[i]['type_broad'] for i in members)
        type_str = ', '.join(f'{t}:{n}' for t, n in type_counts.most_common(3))
        degs = [(g_u.vs[i]['name'], g_u.degree(i)) for i in members]
        top3 = sorted(degs, key=lambda x: -x[1])[:3]
        cluster_rows.append({
            'Cluster': c, 'Size': size,
            'Entity Types': type_str,
            'Key Members': ', '.join(n for n, _ in top3),
        })
    cluster_df = pd.DataFrame(cluster_rows)
    save_latex(cluster_df, f'table7_communities_{tag}',
        caption=f'Community Clusters — {label} (Louvain, modularity={community_results[label]["modularity"]:.4f})',
        label=f'communities_{tag}')

    # Community network figure
    fig_comm = plot_network_plotly(g_d, layout_algo='fr',
        title=f'{label} — Communities', color_attr='community')
    save_fig(fig_comm, f'fig7_community_network_{tag}', width=1000, height=750)

# Cross-corpus cluster alignment
opt1_n = set(G_opt1.vs['name'])
opt2_n = set(G_opt2.vs['name'])
shared_c = opt1_n & opt2_n
if shared_c:
    align_rows = []
    for entity in shared_c:
        idx1 = G_opt1.vs['name'].index(entity)
        idx2 = G_opt2.vs['name'].index(entity)
        align_rows.append({
            'Entity': entity,
            'Opt1 Cluster': G_opt1.vs[idx1]['community'],
            'Opt2 Cluster': G_opt2.vs[idx2]['community'],
        })
    align_df = pd.DataFrame(align_rows).sort_values(['Opt1 Cluster', 'Opt2 Cluster'])
    save_latex(align_df, 'table7b_cluster_alignment',
        caption='Cross-Corpus Cluster Alignment for Shared Entities',
        label='cluster_alignment')

# ═══════════════════════════════════════════════════════════════════
# ANALYSIS 7: Sentiment-Weighted Network
# ═══════════════════════════════════════════════════════════════════
print('\n[7/8] Sentiment-Weighted Network')

# Sentiment breakdown table
sent_rows = []
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    total_edges = g.ecount()
    for sent in ['positive', 'negative', 'neutral']:
        r = sentiment_analysis[label][sent]
        sent_rows.append({
            'Corpus': label, 'Sentiment': sent,
            'Edges': r['edges'],
            'Pct': round(100 * r['edges'] / total_edges, 1),
            'Active Nodes': r['active_nodes'],
            'Density': round(r['density'], 4),
        })
sent_df = pd.DataFrame(sent_rows)
save_latex(sent_df, 'table8_sentiment_breakdown',
    caption='Sentiment-Weighted Network Summary',
    label='sentiment_breakdown', float_format='%.4f')

# Sentiment centrality table
for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
    pos_sg = sentiment_analysis[label]['positive']['subgraph']
    neg_sg = sentiment_analysis[label]['negative']['subgraph']
    pos_deg = {pos_sg.vs[i]['name']: pos_sg.degree(i) for i in range(pos_sg.vcount()) if pos_sg.degree(i) > 0}
    neg_deg = {neg_sg.vs[i]['name']: neg_sg.degree(i) for i in range(neg_sg.vcount()) if neg_sg.degree(i) > 0}
    all_ents = set(pos_deg.keys()) | set(neg_deg.keys())
    rows_s = []
    for entity in all_ents:
        pd_val = pos_deg.get(entity, 0)
        nd_val = neg_deg.get(entity, 0)
        total = pd_val + nd_val
        ratio = (pd_val - nd_val) / total if total > 0 else 0
        rows_s.append({
            'Entity': entity, 'Positive Deg': pd_val,
            'Negative Deg': nd_val, 'Sentiment Ratio': round(ratio, 3),
            'Total': total,
        })
    sc_df = pd.DataFrame(rows_s).sort_values('Total', ascending=False).head(15)
    tag = label.lower().replace(' ', '')
    save_latex(sc_df, f'table8b_sentiment_centrality_{tag}',
        caption=f'Top 15 Entities by Sentiment Network Participation — {label}',
        label=f'sentiment_centrality_{tag}')

save_fig(fig_sent, 'fig8_sentiment_distribution', width=700, height=450)

# ═══════════════════════════════════════════════════════════════════


# Cross-corpus sentiment comparison (from Cell 8)
if 'cross_sent_df' in dir():
    save_latex(cross_sent_df, 'table8c_cross_corpus_sentiment',
        caption='Cross-Corpus Sentiment Comparison for Shared Entities',
        label='cross_corpus_sentiment', float_format='%.3f')
    save_fig(fig_cross_sent, 'fig8b_sentiment_shift', width=900, height=max(400, len(cross_sent_df)*28))

# ANALYSIS 8: Cross-Corpus Synthesis
# ═══════════════════════════════════════════════════════════════════
print('\n[8/8] Cross-Corpus Synthesis')

save_latex(fp_df, 'table9_fingerprint',
    caption='Narrative Structure Fingerprint — Cross-Corpus Comparison',
    label='fingerprint', index=True, float_format='%.4f')

cols = ['Entity', 'Opt1_Eigenvector', 'Opt2_Eigenvector',
        'Opt1_Hub', 'Opt2_Hub', 'Opt1_Authority', 'Opt2_Authority',
        'Opt1_Community', 'Opt2_Community']
master_top20 = master_df.head(20)[cols].copy()
master_top20.columns = ['Entity', 'Opt1 Eigen', 'Opt2 Eigen',
                        'Opt1 Hub', 'Opt2 Hub', 'Opt1 Auth', 'Opt2 Auth',
                        'Opt1 Clust', 'Opt2 Clust']
save_latex(master_top20, 'table9b_master_comparison',
    caption='Shared Entity Master Comparison — Top 20 by Average Eigenvector Centrality',
    label='master_comparison')

# Full network visualizations
save_fig(fig1, 'fig10_network_option1', width=1000, height=750)
save_fig(fig2, 'fig10_network_option2', width=1000, height=750)

# ═══════════════════════════════════════════════════════════════════


# Community detection method comparison (from Cell 7)
if 'leiden_results' in dir():
    comp_rows = []
    for lbl in ['Option 1', 'Option 2']:
        comp_rows.append({'Method': 'Leiden', 'Corpus': lbl,
            'Communities': leiden_results[lbl]['n_communities'],
            'Modularity': round(leiden_results[lbl]['modularity'], 4),
            'Notes': f"res={leiden_results[lbl]['resolution']:.3f}"})
        comp_rows.append({'Method': 'Infomap', 'Corpus': lbl,
            'Communities': infomap_results[lbl]['n_communities'],
            'Modularity': round(infomap_results[lbl]['modularity'], 4),
            'Notes': f"codelength={infomap_results[lbl]['codelength']:.3f}"})
        comp_rows.append({'Method': 'Louvain', 'Corpus': lbl,
            'Communities': louvain_results[lbl]['n_communities'],
            'Modularity': round(louvain_results[lbl]['modularity'], 4),
            'Notes': 'baseline'})
    comp_df = pd.DataFrame(comp_rows)
    save_latex(comp_df, 'table7_method_comparison',
        caption='Community Detection Method Comparison',
        label='community_methods')

# Role equivalence (from Cell 8b)
if 'role_summaries' in dir():
    for label in ['Option 1', 'Option 2']:
        tag = label.lower().replace(' ', '')
        save_latex(role_summaries[label], f'table_role_typology_{tag}',
            caption=f'Structural Role Typology — {label}',
            label=f'role_typology_{tag}')

if 'role_comp_df' in dir():
    save_latex(role_comp_df, 'table_role_cross_corpus',
        caption='Cross-Corpus Structural Role Comparison (Shared Entities)',
        label='role_cross_corpus')

# Role heatmaps and network figures are generated inline in Cell 8b

# Role equivalence figures and non-peripheral entity table (from Cell 8b)
if 'role_profiles' in dir():
    # Role feature heatmaps
    for label in ['Option 1', 'Option 2']:
        df_prof = role_profiles[label]
        n_roles = df_prof['role'].nunique()
        tag = label.lower().replace(' ', '')
        
        X = StandardScaler().fit_transform(df_prof[profile_features].fillna(0))
        df_norm = pd.DataFrame(X, columns=profile_features)
        df_norm['role'] = df_prof['role'].values
        role_means = df_norm.groupby('role')[profile_features].mean()
        
        fig_heat = go.Figure(data=go.Heatmap(
            z=role_means.values,
            x=[f.replace('_', ' ').title() for f in profile_features],
            y=[f'Role {r}' for r in role_means.index],
            colorscale='RdBu_r', zmid=0,
            text=np.round(role_means.values, 2), texttemplate='%{text}',
        ))
        fig_heat.update_layout(
            title=f'{label} — Structural Role Feature Profiles (standardized)',
            width=900, height=300 + n_roles * 40,
            xaxis_title='Feature', yaxis_title='Role',
        )
        save_fig(fig_heat, f'fig_role_heatmap_{tag}', width=900, height=300 + n_roles * 40)
    
    # Role network figures
    for label, g in [('Option 1', G_opt1), ('Option 2', G_opt2)]:
        tag = label.lower().replace(' ', '')
        fig_role = plot_network_plotly(g, layout_algo='fr',
            title=f'{label} - Structural Roles', color_attr='role')
        save_fig(fig_role, f'fig_role_network_{tag}', width=1000, height=750)
    
    # Non-peripheral entity tables
    for label in ['Option 1', 'Option 2']:
        df_prof = role_profiles[label]
        tag = label.lower().replace(' ', '')
        peripheral_role = df_prof['role'].value_counts().idxmax()
        
        non_periph = df_prof[df_prof['role'] != peripheral_role].copy()
        non_periph = non_periph.sort_values(['role', 'eigenvector'], ascending=[True, False])
        non_periph['degree'] = non_periph['in_degree'] + non_periph['out_degree']
        
        # Add descriptive role label
        def label_role(r):
            sub = non_periph[non_periph['role'] == r]
            if sub['hub'].mean() > 0.3 or sub['betweenness'].mean() > 1000:
                return 'Protagonist'
            elif sub['hub'].mean() > 0.05 or sub['betweenness'].mean() > 100:
                return 'Secondary'
            return 'Supporting'
        
        non_periph['role_label'] = non_periph['role'].map(
            {r: label_role(r) for r in non_periph['role'].unique()})
        
        export_cols = ['name', 'type', 'role', 'role_label', 'degree',
                       'hub', 'authority', 'betweenness', 'sentiment_ratio']
        df_export = non_periph[export_cols].copy()
        df_export.columns = ['Entity', 'Type', 'Role', 'Role Label', 'Degree',
                             'Hub', 'Authority', 'Betweenness', 'Sent. Ratio']
        
        save_latex(df_export, f'table_nonperipheral_entities_{tag}',
            caption=f'Non-Peripheral Entities by Structural Role — {label}',
            label=f'nonperipheral_{tag}')


# Directional sentiment tables and figures (from Cell 8)
if 'directional_sentiment' in dir():
    from sklearn.preprocessing import StandardScaler  # ensure available
    
    for label in ['Option 1', 'Option 2']:
        tag = label.lower().replace(' ', '')
        save_latex(directional_sentiment[label].head(15),
            f'table_directional_sentiment_{tag}',
            caption=f'Top 15 Entities by Directional Sentiment — {label}',
            label=f'directional_sentiment_{tag}')

if 'cross_dir_df' in dir():
    save_latex(cross_dir_df, 'table_cross_corpus_directional_sentiment',
        caption='Cross-Corpus Directional Sentiment Comparison (Agency and Patient Shifts)',
        label='cross_directional_sentiment')

if 'fig_dir_sent' in dir():
    save_fig(fig_dir_sent, 'fig_directional_sentiment_shift', width=1100,
        height=max(500, len(cross_dir_df) * 28))


# Summary
# ═══════════════════════════════════════════════════════════════════
print('\n' + '='*70)
figs = [f for f in os.listdir(FIG_DIR) if f.endswith('.png')]
tabs = [f for f in os.listdir(TAB_DIR) if f.endswith('.tex')]
print(f'  Export complete!')
print(f'  Figures: {len(figs)} PNG files in {FIG_DIR}/')
print(f'  Tables:  {len(tabs)} LaTeX files in {TAB_DIR}/')
print(f'\n  Figures:')
for f in sorted(figs):
    size_kb = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
    print(f'    {f:<50} ({size_kb:.0f} KB)')
print(f'\n  Tables:')
for f in sorted(tabs):
    print(f'    {f}')
print('='*70)